<a href="https://colab.research.google.com/github/Ganasa18/belajar-tensorflow/blob/main/lab_action_classifier_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# =========================================================
# CELL 1 — GOOGLE DRIVE + TRAINING PATH SETUP
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

from google.colab import drive

import os


# =========================================================
# 1. MOUNT GOOGLE DRIVE
# =========================================================

DRIVE_MOUNT = "/content/drive"

drive.mount(
    DRIVE_MOUNT,
    force_remount=False
)

assert os.path.exists(
    f"{DRIVE_MOUNT}/MyDrive"
), "Google Drive belum mounted"


# =========================================================
# 2. STORAGE DIRECTORY
# =========================================================

SAVE_DIR = (
    f"{DRIVE_MOUNT}/MyDrive/action_classifier"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


# =========================================================
# 3. SUBDIRECTORIES
# =========================================================

RAW_DIR = (
    f"{SAVE_DIR}/raw"
)

PROCESSED_DIR = (
    f"{SAVE_DIR}/processed"
)

TEACHER_DIR = (
    f"{SAVE_DIR}/teacher"
)

TRUSTED_DIR = (
    f"{SAVE_DIR}/trusted"
)

REPAIR_DIR = (
    f"{SAVE_DIR}/repair"
)

BLIND_TEST_DIR = (
    f"{SAVE_DIR}/blind_test"
)

MODEL_DIR = (
    f"{SAVE_DIR}/models"
)

ONNX_DIR = (
    f"{MODEL_DIR}/onnx"
)

REPORT_DIR = (
    f"{SAVE_DIR}/reports"
)


for directory in [
    RAW_DIR,
    PROCESSED_DIR,
    TEACHER_DIR,
    TRUSTED_DIR,
    REPAIR_DIR,
    BLIND_TEST_DIR,
    MODEL_DIR,
    ONNX_DIR,
    REPORT_DIR,
]:

    os.makedirs(
        directory,
        exist_ok=True
    )


# =========================================================
# 4. DATASET PATHS
# =========================================================

MANUAL_SEED_PATH = (
    f"{RAW_DIR}/manual_seed_v1.jsonl"
)

PUBLIC_RAW_PATH = (
    f"{RAW_DIR}/public_seed_raw.jsonl"
)

NORMALIZED_DATASET_PATH = (
    f"{PROCESSED_DIR}/normalized_dataset_v1.jsonl"
)

DEDUP_DATASET_PATH = (
    f"{PROCESSED_DIR}/dedup_dataset_v1.jsonl"
)

TEACHER_LABELED_PATH = (
    f"{TEACHER_DIR}/teacher_labeled_v1.jsonl"
)

TEACHER_REJECTED_PATH = (
    f"{TEACHER_DIR}/teacher_rejected_v1.jsonl"
)

TRUSTED_TRAIN_PATH = (
    f"{TRUSTED_DIR}/action_classifier_trusted_v1.jsonl"
)

REPAIR_DATASET_PATH = (
    f"{REPAIR_DIR}/targeted_repair_v1.jsonl"
)

BLIND_TEST_PATH = (
    f"{BLIND_TEST_DIR}/blind_test_v1.jsonl"
)


# =========================================================
# 5. MODEL OUTPUT PATHS
# =========================================================

MODEL_PATH = (
    f"{MODEL_DIR}/action_classifier.joblib"
)

BEST_MODEL_PATH = (
    f"{MODEL_DIR}/action_classifier_best.joblib"
)

MLB_PATH = (
    f"{MODEL_DIR}/multilabel_binarizer.joblib"
)

ONNX_MODEL_PATH = (
    f"{ONNX_DIR}/action_classifier.onnx"
)


# =========================================================
# 6. REPORT PATHS
# =========================================================

METRICS_PATH = (
    f"{REPORT_DIR}/training_metrics.json"
)

PREDICTIONS_PATH = (
    f"{REPORT_DIR}/test_predictions.csv"
)

ERROR_ANALYSIS_PATH = (
    f"{REPORT_DIR}/error_analysis.csv"
)

LABEL_DISTRIBUTION_PATH = (
    f"{REPORT_DIR}/label_distribution.csv"
)

ONNX_PARITY_PATH = (
    f"{REPORT_DIR}/onnx_parity.json"
)

BENCHMARK_PATH = (
    f"{REPORT_DIR}/latency_benchmark.json"
)


# =========================================================
# 7. STATUS
# =========================================================

print()
print("=" * 70)
print("MODEL 01 — ACTION CLASSIFIER STORAGE")
print("=" * 70)

print(
    "SAVE_DIR       :",
    SAVE_DIR
)

print(
    "Manual seed    :",
    MANUAL_SEED_PATH
)

print(
    "Trusted train  :",
    TRUSTED_TRAIN_PATH
)

print(
    "Repair dataset :",
    REPAIR_DATASET_PATH
)

print(
    "Model output   :",
    MODEL_PATH
)

print(
    "Best model     :",
    BEST_MODEL_PATH
)

print(
    "ONNX model     :",
    ONNX_MODEL_PATH
)

print(
    "Metrics        :",
    METRICS_PATH
)

print("=" * 70)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

MODEL 01 — ACTION CLASSIFIER STORAGE
SAVE_DIR       : /content/drive/MyDrive/action_classifier
Manual seed    : /content/drive/MyDrive/action_classifier/raw/manual_seed_v1.jsonl
Trusted train  : /content/drive/MyDrive/action_classifier/trusted/action_classifier_trusted_v1.jsonl
Repair dataset : /content/drive/MyDrive/action_classifier/repair/targeted_repair_v1.jsonl
Model output   : /content/drive/MyDrive/action_classifier/models/action_classifier.joblib
Best model     : /content/drive/MyDrive/action_classifier/models/action_classifier_best.joblib
ONNX model     : /content/drive/MyDrive/action_classifier/models/onnx/action_classifier.onnx
Metrics        : /content/drive/MyDrive/action_classifier/reports/training_metrics.json


In [4]:
# =========================================================
# CELL 2 — DEPENDENCIES + IMPORTS
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

!pip -q install \
    pandas \
    numpy \
    scikit-learn \
    datasets \
    huggingface_hub \
    tqdm \
    matplotlib \
    joblib \
    requests \
    skl2onnx \
    onnx \
    onnxruntime

print("Dependencies installed.")


# =========================================================
# IMPORTS
# =========================================================

import os
import re
import json
import time
import random
import hashlib
import warnings

import requests
import joblib

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.model_selection import (
    train_test_split
)

from sklearn.preprocessing import (
    MultiLabelBinarizer
)

from sklearn.metrics import (
    classification_report,
    f1_score,
    accuracy_score,
    hamming_loss,
)


# =========================================================
# RANDOM SEED
# =========================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

warnings.filterwarnings(
    "ignore"
)


# =========================================================
# STATUS
# =========================================================

print()
print("=" * 70)
print("ACTION CLASSIFIER — ENVIRONMENT")
print("=" * 70)

print(
    "Random seed :",
    SEED
)

print(
    "NumPy       :",
    np.__version__
)

print(
    "Pandas      :",
    pd.__version__
)

print("=" * 70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 48.9 MB/s eta 0:00:00
Dependencies installed.

ACTION CLASSIFIER — ENVIRONMENT
Random seed : 42
NumPy       : 2.1.3
Pandas      : 2.2.3


In [5]:
# =========================================================
# CELL 3 — ACTION TAXONOMY + DATASET SCHEMA
# MODEL 01 — ACTION CLASSIFIER
# =========================================================


# =========================================================
# 1. ACTION LABELS
# =========================================================

LABELS = [
    "READ",
    "WRITE",
    "DELETE",
    "EXECUTE",
    "NETWORK",
    "INSTALL",
    "PRIVILEGED",
    "SYSTEM_CHANGE",
]


LABEL_TO_ID = {
    label: index
    for index, label in enumerate(LABELS)
}

ID_TO_LABEL = {
    index: label
    for label, index in LABEL_TO_ID.items()
}


# =========================================================
# 2. DATASET COLUMNS
# =========================================================

DATASET_COLUMNS = [
    "command",
    "description",
    "actions",
    "confidence",
    "ambiguous",
    "source",
    "split_origin",
]


# =========================================================
# 3. VALIDATION FUNCTION
# =========================================================

def validate_actions(actions):

    if not isinstance(
        actions,
        list
    ):
        return False

    if len(actions) == 0:
        return False

    if len(actions) != len(set(actions)):
        return False

    for action in actions:

        if action not in LABELS:
            return False

    return True


# =========================================================
# 4. SCHEMA
# =========================================================

SCHEMA = {

    "model": (
        "action_classifier"
    ),

    "version": (
        "v1"
    ),

    "task": (
        "multi_label_classification"
    ),

    "labels": (
        LABELS
    ),

    "columns": (
        DATASET_COLUMNS
    ),
}


SCHEMA_PATH = (
    f"{PROCESSED_DIR}/dataset_schema_v1.json"
)


with open(
    SCHEMA_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        SCHEMA,
        f,
        indent=2,
        ensure_ascii=False
    )


# =========================================================
# 5. STATUS
# =========================================================

print()
print("=" * 70)
print("ACTION CLASSIFIER — TAXONOMY")
print("=" * 70)

for index, label in enumerate(LABELS):

    print(
        f"{index:2} -> {label}"
    )


print()
print(
    "Total labels :",
    len(LABELS)
)

print(
    "Task         :",
    "MULTI-LABEL"
)

print(
    "Schema       :",
    SCHEMA_PATH
)

print("=" * 70)


ACTION CLASSIFIER — TAXONOMY
 0 -> READ
 1 -> WRITE
 2 -> DELETE
 3 -> EXECUTE
 4 -> NETWORK
 5 -> INSTALL
 6 -> PRIVILEGED
 7 -> SYSTEM_CHANGE

Total labels : 8
Task         : MULTI-LABEL
Schema       : /content/drive/MyDrive/action_classifier/processed/dataset_schema_v1.json


In [6]:
# =========================================================
# CELL 4 — TEACHER API SETUP
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

import requests
import time
import re
import json

from google.colab import userdata


# =========================================================
# API CONFIG
# =========================================================

BASE_URL = (
    "https://api.deepseek.com/chat/completions"
)

API_KEY = userdata.get(
    "DEEPSEEK"
)

MODEL = "deepseek-v4-flash"

TEMPERATURE = 0

TIMEOUT_CONNECT = 10
TIMEOUT_READ = 45


if not API_KEY:

    raise RuntimeError(
        "DEEPSEEK API key tidak ditemukan "
        "di Colab Secrets."
    )


# =========================================================
# TEACHER CLASSIFICATION PROMPT
# =========================================================

TEACHER_SYSTEM_PROMPT = """
You are a strict shell-command action classifier.

Your task is NOT to execute commands.

Your task is NOT to judge whether a command is malicious.

Your task is ONLY to identify what ACTIONS the command
would perform if executed.

The classification is MULTI-LABEL.

A command may have one or multiple action labels.

Allowed labels:

READ
WRITE
DELETE
EXECUTE
NETWORK
INSTALL
PRIVILEGED
SYSTEM_CHANGE


=========================================================
READ
=========================================================

The command reads, displays, lists, inspects, queries,
or retrieves local information without intentionally
modifying it.

Examples:

cat /etc/os-release
-> READ

ls -la /tmp
-> READ

systemctl status ssh
-> READ


=========================================================
WRITE
=========================================================

The command creates, copies, moves, overwrites,
appends, downloads, or otherwise writes data
to local storage.

Examples:

echo hello > output.txt
-> WRITE

cp source.txt backup.txt
-> READ + WRITE

curl <TEST_URL> -o file
-> NETWORK + WRITE


=========================================================
DELETE
=========================================================

The command removes files, directories, records,
or other persistent data.

Examples:

rm test.txt
-> DELETE

rm -r <TEMP_DIR>
-> DELETE


=========================================================
EXECUTE
=========================================================

The command launches, runs, evaluates, invokes,
or restarts executable code, programs, scripts,
commands, or services.

Examples:

python app.py
-> EXECUTE

bash script.sh
-> EXECUTE

sudo systemctl restart nginx
-> EXECUTE + PRIVILEGED + SYSTEM_CHANGE


=========================================================
NETWORK
=========================================================

The command sends, receives, downloads, uploads,
queries, connects to, scans, or otherwise interacts
with a network or remote host.

Examples:

curl <TEST_URL>
-> NETWORK

ping <LAB_HOST>
-> NETWORK

wget <TEST_URL> -O file
-> NETWORK + WRITE


=========================================================
INSTALL
=========================================================

The command installs, adds, upgrades, or removes
software packages or software dependencies.

Examples:

pip install requests
-> NETWORK + INSTALL + SYSTEM_CHANGE

sudo apt install nginx
-> NETWORK + INSTALL + PRIVILEGED + SYSTEM_CHANGE


=========================================================
PRIVILEGED
=========================================================

The command explicitly requests elevated privileges
or performs an operation requiring an elevated
privilege boundary.

Examples:

sudo apt update
-> NETWORK + PRIVILEGED + SYSTEM_CHANGE

sudo systemctl restart ssh
-> EXECUTE + PRIVILEGED + SYSTEM_CHANGE


=========================================================
SYSTEM_CHANGE
=========================================================

The command modifies system state, configuration,
packages, services, permissions, users, system files,
mounts, firewall state, or other operating-system
configuration.

Examples:

chmod 600 file
-> SYSTEM_CHANGE

sudo systemctl restart nginx
-> EXECUTE + PRIVILEGED + SYSTEM_CHANGE

sudo apt install nginx
-> NETWORK + INSTALL + PRIVILEGED + SYSTEM_CHANGE


=========================================================
IMPORTANT RULES
=========================================================

1. Return every action clearly implied by the command.

2. Do NOT classify based on whether the command is
   good, bad, suspicious, malicious, or dangerous.

3. Risk classification belongs to another model.

4. Focus only on observable command behavior.

5. Do not invent actions that are not implied.

6. If the command is genuinely unclear or cannot be
   reliably classified, set:

   "ambiguous": true

7. Confidence must reflect classification certainty.

8. Shell chaining must be classified across the
   entire command.

Example:

cat input.txt | curl -X POST <TEST_URL> -d @-

-> READ + NETWORK


=========================================================
OUTPUT
=========================================================

Return ONLY valid JSON.

Schema:

{
  "actions": ["LABEL"],
  "confidence": 0.95,
  "ambiguous": false
}

actions:
- must be a JSON array
- may contain one or multiple allowed labels
- must not contain duplicates

confidence:
- number from 0 to 1

ambiguous:
- true or false

Do not include explanations.
Do not include markdown.
Do not include reasoning.
"""


# =========================================================
# TEACHER CALL
# =========================================================

def call_teacher(command):

    headers = {

        "Authorization":
            f"Bearer {API_KEY}",

        "Content-Type":
            "application/json",
    }


    payload = {

        "model":
            MODEL,

        "temperature":
            TEMPERATURE,

        "thinking": {
            "type": "disabled"
        },

        "response_format": {
            "type": "json_object"
        },

        "messages": [

            {
                "role": "system",
                "content": TEACHER_SYSTEM_PROMPT,
            },

            {
                "role": "user",
                "content": command,
            },
        ],
    }


    start = time.time()


    response = requests.post(

        BASE_URL,

        headers=headers,

        json=payload,

        timeout=(
            TIMEOUT_CONNECT,
            TIMEOUT_READ
        ),
    )


    latency = (
        time.time()
        - start
    )


    response.raise_for_status()


    data = response.json()


    content = (
        data["choices"][0]
            ["message"]
            ["content"]
    )


    return content, latency


# =========================================================
# PARSER
# =========================================================

def parse_teacher_output(text):

    if not text:

        raise ValueError(
            "Teacher response kosong."
        )


    text = text.strip()


    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.I
    )


    text = re.sub(
        r"\s*```$",
        "",
        text
    )


    match = re.search(
        r"\{[\s\S]*?\}",
        text
    )


    if not match:

        raise ValueError(
            "JSON tidak ditemukan: "
            + text[:300]
        )


    data = json.loads(
        match.group()
    )


    actions = data.get(
        "actions",
        []
    )


    confidence = float(
        data.get(
            "confidence"
        )
    )


    ambiguous = data.get(
        "ambiguous"
    )


    # =====================================================
    # VALIDATE ACTIONS
    # =====================================================

    if not isinstance(
        actions,
        list
    ):

        raise ValueError(
            "actions harus berupa list."
        )


    actions = [

        str(action)
        .strip()
        .upper()

        for action in actions
    ]


    # Remove duplicates while preserving order

    actions = list(
        dict.fromkeys(actions)
    )


    if not actions:

        raise ValueError(
            "actions kosong."
        )


    for action in actions:

        if action not in LABELS:

            raise ValueError(
                f"Invalid action label: {action}"
            )


    # =====================================================
    # VALIDATE CONFIDENCE
    # =====================================================

    if not 0 <= confidence <= 1:

        raise ValueError(
            f"Invalid confidence: {confidence}"
        )


    # =====================================================
    # VALIDATE AMBIGUOUS
    # =====================================================

    if not isinstance(
        ambiguous,
        bool
    ):

        raise ValueError(
            "ambiguous harus boolean."
        )


    return {

        "actions":
            actions,

        "confidence":
            confidence,

        "ambiguous":
            ambiguous,
    }


# =========================================================
# STATUS
# =========================================================

print()
print("=" * 70)
print("ACTION CLASSIFIER — TEACHER API")
print("=" * 70)

print(
    "Model          :",
    MODEL
)

print(
    "API key        :",
    "OK"
)

print(
    "Labels         :",
    len(LABELS)
)

print(
    "call_teacher() :",
    "OK"
)

print(
    "parser         :",
    "OK"
)

print("=" * 70)


ACTION CLASSIFIER — TEACHER API
Model          : deepseek-v4-flash
API key        : OK
Labels         : 8
call_teacher() : OK
parser         : OK


In [7]:
# =========================================================
# CELL 5 — TEACHER API SANITY TEST
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

SANITY_CASES = [
    {
        "command": "cat /etc/os-release",
        "expected": ["READ"],
    },
    {
        "command": "ls -la /tmp",
        "expected": ["READ"],
    },
    {
        "command": "echo hello > output.txt",
        "expected": ["WRITE"],
    },
    {
        "command": "cp source.txt backup.txt",
        "expected": ["READ", "WRITE"],
    },
    {
        "command": "rm test.txt",
        "expected": ["DELETE"],
    },
    {
        "command": "python app.py",
        "expected": ["EXECUTE"],
    },
    {
        "command": "curl https://example.com/file -o file",
        "expected": ["NETWORK", "WRITE"],
    },
    {
        "command": "wget https://example.com/archive.tar.gz",
        "expected": ["NETWORK", "WRITE"],
    },
    {
        "command": "sudo apt update",
        "expected": [
            "NETWORK",
            "PRIVILEGED",
            "SYSTEM_CHANGE",
        ],
    },
    {
        "command": "sudo apt install nginx",
        "expected": [
            "NETWORK",
            "INSTALL",
            "PRIVILEGED",
            "SYSTEM_CHANGE",
        ],
    },
    {
        "command": "systemctl status ssh",
        "expected": ["READ"],
    },
    {
        "command": "sudo systemctl restart nginx",
        "expected": [
            "EXECUTE",
            "PRIVILEGED",
            "SYSTEM_CHANGE",
        ],
    },
]


def normalize_label_set(labels):

    return set(
        str(label).strip().upper()
        for label in labels
    )


results = []


print()
print("=" * 90)
print("ACTION CLASSIFIER — TEACHER SANITY TEST")
print("=" * 90)


for index, case in enumerate(
    SANITY_CASES,
    start=1
):

    command = case["command"]
    expected = case["expected"]

    try:

        raw_output, latency = call_teacher(
            command
        )

        parsed = parse_teacher_output(
            raw_output
        )

        predicted = parsed["actions"]

        exact_match = (
            normalize_label_set(predicted)
            ==
            normalize_label_set(expected)
        )

        row = {
            "index": index,
            "command": command,
            "expected": expected,
            "predicted": predicted,
            "confidence": parsed["confidence"],
            "ambiguous": parsed["ambiguous"],
            "latency": latency,
            "exact_match": exact_match,
            "error": None,
        }

    except Exception as e:

        row = {
            "index": index,
            "command": command,
            "expected": expected,
            "predicted": None,
            "confidence": None,
            "ambiguous": None,
            "latency": None,
            "exact_match": False,
            "error": str(e),
        }


    results.append(row)


    print()
    print(
        f"[{index}/{len(SANITY_CASES)}]",
        command
    )

    print(
        "Expected :",
        expected
    )

    print(
        "Predicted:",
        row["predicted"]
    )

    print(
        "Confidence:",
        row["confidence"]
    )

    print(
        "Match:",
        row["exact_match"]
    )


SANITY_DF = pd.DataFrame(
    results
)


print()
print("=" * 90)

successful = (
    SANITY_DF["error"].isna().sum()
)

exact = (
    SANITY_DF["exact_match"].sum()
)

print(
    "Successful calls :",
    successful,
    "/",
    len(SANITY_DF)
)

print(
    "Exact matches    :",
    exact,
    "/",
    len(SANITY_DF)
)

print(
    "Exact accuracy   :",
    round(
        exact / len(SANITY_DF),
        4
    )
)

print("=" * 90)


ACTION CLASSIFIER — TEACHER SANITY TEST

[1/12] cat /etc/os-release
Expected : ['READ']
Predicted: ['READ']
Confidence: 1.0
Match: True

[2/12] ls -la /tmp
Expected : ['READ']
Predicted: ['READ']
Confidence: 1.0
Match: True

[3/12] echo hello > output.txt
Expected : ['WRITE']
Predicted: ['WRITE']
Confidence: 1.0
Match: True

[4/12] cp source.txt backup.txt
Expected : ['READ', 'WRITE']
Predicted: ['READ', 'WRITE']
Confidence: 1.0
Match: True

[5/12] rm test.txt
Expected : ['DELETE']
Predicted: ['DELETE']
Confidence: 1.0
Match: True

[6/12] python app.py
Expected : ['EXECUTE']
Predicted: ['EXECUTE']
Confidence: 1.0
Match: True

[7/12] curl https://example.com/file -o file
Expected : ['NETWORK', 'WRITE']
Predicted: ['NETWORK', 'WRITE']
Confidence: 1.0
Match: True

[8/12] wget https://example.com/archive.tar.gz
Expected : ['NETWORK', 'WRITE']
Predicted: ['NETWORK', 'WRITE']
Confidence: 1.0
Match: True

[9/12] sudo apt update
Expected : ['NETWORK', 'PRIVILEGED', 'SYSTEM_CHANGE']
Predicted:

In [8]:
# =========================================================
# CELL 6 — CREATE MANUAL SEED DATASET
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

MANUAL_SAMPLES = [

    # =====================================================
    # READ
    # =====================================================

    {
        "command": "cat /etc/os-release",
        "description": "Read operating system release information",
        "actions": ["READ"],
    },

    {
        "command": "ls -la /tmp",
        "description": "List files in temporary directory",
        "actions": ["READ"],
    },

    {
        "command": "pwd",
        "description": "Print current working directory",
        "actions": ["READ"],
    },

    {
        "command": "whoami",
        "description": "Display current user",
        "actions": ["READ"],
    },

    {
        "command": "systemctl status ssh",
        "description": "Inspect SSH service status",
        "actions": ["READ"],
    },


    # =====================================================
    # WRITE
    # =====================================================

    {
        "command": "echo hello > output.txt",
        "description": "Write text into a file",
        "actions": ["WRITE"],
    },

    {
        "command": "touch test.txt",
        "description": "Create an empty file",
        "actions": ["WRITE"],
    },

    {
        "command": "cp source.txt backup.txt",
        "description": "Read source file and create a copy",
        "actions": ["READ", "WRITE"],
    },


    # =====================================================
    # DELETE
    # =====================================================

    {
        "command": "rm test.txt",
        "description": "Delete a file",
        "actions": ["DELETE"],
    },

    {
        "command": "rm -r temp_folder",
        "description": "Delete a directory recursively",
        "actions": ["DELETE"],
    },


    # =====================================================
    # EXECUTE
    # =====================================================

    {
        "command": "python app.py",
        "description": "Execute a Python program",
        "actions": ["EXECUTE"],
    },

    {
        "command": "bash script.sh",
        "description": "Execute a shell script",
        "actions": ["EXECUTE"],
    },


    # =====================================================
    # NETWORK
    # =====================================================

    {
        "command": "curl https://example.com",
        "description": "Request data from a remote HTTP server",
        "actions": ["NETWORK"],
    },

    {
        "command": "ping example.com",
        "description": "Send network echo requests to remote host",
        "actions": ["NETWORK"],
    },

    {
        "command": "curl https://example.com/file -o file",
        "description": "Download a remote file to local storage",
        "actions": ["NETWORK", "WRITE"],
    },


    # =====================================================
    # INSTALL
    # =====================================================

    {
        "command": "pip install requests",
        "description": "Install a Python package",
        "actions": [
            "NETWORK",
            "INSTALL",
            "SYSTEM_CHANGE",
        ],
    },

    {
        "command": "npm install express",
        "description": "Install a Node.js dependency",
        "actions": [
            "NETWORK",
            "INSTALL",
            "SYSTEM_CHANGE",
        ],
    },


    # =====================================================
    # PRIVILEGED + SYSTEM_CHANGE
    # =====================================================

    {
        "command": "sudo apt update",
        "description": "Update system package repository metadata",
        "actions": [
            "NETWORK",
            "PRIVILEGED",
            "SYSTEM_CHANGE",
        ],
    },

    {
        "command": "sudo apt install nginx",
        "description": "Install nginx using system package manager",
        "actions": [
            "NETWORK",
            "INSTALL",
            "PRIVILEGED",
            "SYSTEM_CHANGE",
        ],
    },

    {
        "command": "chmod 600 config.txt",
        "description": "Change file permissions",
        "actions": [
            "SYSTEM_CHANGE",
        ],
    },

    {
        "command": "sudo systemctl restart nginx",
        "description": "Restart nginx system service",
        "actions": [
            "EXECUTE",
            "PRIVILEGED",
            "SYSTEM_CHANGE",
        ],
    },


    # =====================================================
    # MULTI-ACTION / CHAINED
    # =====================================================

    {
        "command": "cat input.txt | curl -X POST https://example.com -d @-",
        "description": "Read local data and send it to a remote server",
        "actions": [
            "READ",
            "NETWORK",
        ],
    },

    {
        "command": "wget https://example.com/app.py -O app.py && python app.py",
        "description": "Download a Python program and execute it",
        "actions": [
            "NETWORK",
            "WRITE",
            "EXECUTE",
        ],
    },
]


manual_df = pd.DataFrame(
    MANUAL_SAMPLES
)


manual_df["confidence"] = 1.0

manual_df["ambiguous"] = False

manual_df["source"] = (
    "manual_seed"
)

manual_df["split_origin"] = (
    "manual"
)


print()
print("=" * 70)
print("ACTION CLASSIFIER — MANUAL SEED")
print("=" * 70)

print(
    "Samples:",
    len(manual_df)
)

print()

display(
    manual_df[
        [
            "command",
            "actions"
        ]
    ]
)

print("=" * 70)


ACTION CLASSIFIER — MANUAL SEED
Samples: 23



,command,actions
0,cat /etc/os-release,[READ]
1,ls -la /tmp,[READ]
2,pwd,[READ]
3,whoami,[READ]
4,systemctl status ssh,[READ]
5,echo hello > output.txt,[WRITE]
6,touch test.txt,[WRITE]
7,cp source.txt backup.txt,"[READ, WRITE]"
8,rm test.txt,[DELETE]
9,rm -r temp_folder,[DELETE]


In [9]:
# =========================================================
# CELL 7 — MANUAL SEED VALIDATION
# MODEL 01 — ACTION CLASSIFIER
# =========================================================


# =========================================================
# 1. BASIC VALIDATION
# =========================================================

validation_errors = []


for index, row in manual_df.iterrows():

    command = str(
        row["command"]
    ).strip()

    actions = row[
        "actions"
    ]


    if not command:

        validation_errors.append(
            {
                "row": index,
                "error": "empty_command",
            }
        )


    if not validate_actions(
        actions
    ):

        validation_errors.append(
            {
                "row": index,
                "error": (
                    f"invalid_actions: {actions}"
                ),
            }
        )


# =========================================================
# 2. DUPLICATE COMMAND CHECK
# =========================================================

duplicate_mask = (
    manual_df["command"]
    .str.strip()
    .str.lower()
    .duplicated(
        keep=False
    )
)


duplicate_rows = (
    manual_df[
        duplicate_mask
    ]
)


# =========================================================
# 3. LABEL COVERAGE
# =========================================================

label_counts = {
    label: 0
    for label in LABELS
}


for actions in manual_df["actions"]:

    for label in actions:

        label_counts[label] += 1


label_distribution_df = pd.DataFrame(
    [
        {
            "label": label,
            "count": count,
        }

        for label, count
        in label_counts.items()
    ]
)


# =========================================================
# 4. STATUS
# =========================================================

print()
print("=" * 70)
print("ACTION CLASSIFIER — MANUAL VALIDATION")
print("=" * 70)


print(
    "Validation errors :",
    len(validation_errors)
)

print(
    "Duplicate commands:",
    len(duplicate_rows)
)


print()
print("LABEL COVERAGE")
print("-" * 70)

display(
    label_distribution_df
)


if validation_errors:

    print()
    print("ERRORS:")

    display(
        pd.DataFrame(
            validation_errors
        )
    )


if len(duplicate_rows) > 0:

    print()
    print("DUPLICATES:")

    display(
        duplicate_rows[
            [
                "command",
                "actions"
            ]
        ]
    )


assert (
    len(validation_errors) == 0
), "Manual dataset memiliki validation error."


assert (
    len(duplicate_rows) == 0
), "Manual dataset memiliki duplicate command."


missing_labels = [

    label

    for label, count
    in label_counts.items()

    if count == 0
]


assert (
    not missing_labels
), (
    "Label belum ter-cover: "
    + str(missing_labels)
)


print()
print(
    "Manual seed validation: PASS"
)

print("=" * 70)


ACTION CLASSIFIER — MANUAL VALIDATION
Validation errors : 0
Duplicate commands: 0

LABEL COVERAGE
----------------------------------------------------------------------


,label,count
0,READ,7
1,WRITE,5
2,DELETE,2
3,EXECUTE,4
4,NETWORK,9
5,INSTALL,3
6,PRIVILEGED,3
7,SYSTEM_CHANGE,6



Manual seed validation: PASS


In [10]:
# =========================================================
# CELL 8 — SAVE MANUAL SEED + SANITY REPORT
# MODEL 01 — ACTION CLASSIFIER
# =========================================================


# =========================================================
# 1. SAVE MANUAL DATASET
# =========================================================

manual_df.to_json(
    MANUAL_SEED_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)


# =========================================================
# 2. SAVE LABEL DISTRIBUTION
# =========================================================

label_distribution_df.to_csv(
    LABEL_DISTRIBUTION_PATH,
    index=False
)


# =========================================================
# 3. SAVE TEACHER SANITY RESULTS
# =========================================================

TEACHER_SANITY_PATH = (
    f"{REPORT_DIR}/teacher_sanity_test.csv"
)


SANITY_SAVE_DF = (
    SANITY_DF.copy()
)


SANITY_SAVE_DF[
    "expected"
] = SANITY_SAVE_DF[
    "expected"
].apply(
    json.dumps
)


SANITY_SAVE_DF[
    "predicted"
] = SANITY_SAVE_DF[
    "predicted"
].apply(
    lambda x:
        json.dumps(x)
        if isinstance(x, list)
        else x
)


SANITY_SAVE_DF.to_csv(
    TEACHER_SANITY_PATH,
    index=False
)


# =========================================================
# 4. SUMMARY
# =========================================================

teacher_exact_accuracy = (
    SANITY_DF[
        "exact_match"
    ].mean()
)


teacher_avg_confidence = (
    SANITY_DF[
        "confidence"
    ].dropna().mean()
)


teacher_avg_latency = (
    SANITY_DF[
        "latency"
    ].dropna().mean()
)


SUMMARY = {

    "manual_samples":
        int(
            len(manual_df)
        ),

    "teacher_sanity_samples":
        int(
            len(SANITY_DF)
        ),

    "teacher_exact_accuracy":
        float(
            teacher_exact_accuracy
        ),

    "teacher_avg_confidence":
        float(
            teacher_avg_confidence
        ),

    "teacher_avg_latency_sec":
        float(
            teacher_avg_latency
        ),
}


TEACHER_SANITY_SUMMARY_PATH = (
    f"{REPORT_DIR}/teacher_sanity_summary.json"
)


with open(
    TEACHER_SANITY_SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        SUMMARY,
        f,
        indent=2,
        ensure_ascii=False
    )


# =========================================================
# 5. STATUS
# =========================================================

print()
print("=" * 70)
print("ACTION CLASSIFIER — SEED DATA SAVED")
print("=" * 70)

print(
    "Manual seed :",
    MANUAL_SEED_PATH
)

print(
    "Label report:",
    LABEL_DISTRIBUTION_PATH
)

print(
    "Teacher test:",
    TEACHER_SANITY_PATH
)

print(
    "Summary     :",
    TEACHER_SANITY_SUMMARY_PATH
)

print()
print(
    "Manual samples          :",
    SUMMARY[
        "manual_samples"
    ]
)

print(
    "Teacher exact accuracy  :",
    round(
        SUMMARY[
            "teacher_exact_accuracy"
        ],
        4
    )
)

print(
    "Teacher avg confidence  :",
    round(
        SUMMARY[
            "teacher_avg_confidence"
        ],
        4
    )
)

print(
    "Teacher avg latency (s) :",
    round(
        SUMMARY[
            "teacher_avg_latency_sec"
        ],
        3
    )
)

print("=" * 70)


ACTION CLASSIFIER — SEED DATA SAVED
Manual seed : /content/drive/MyDrive/action_classifier/raw/manual_seed_v1.jsonl
Label report: /content/drive/MyDrive/action_classifier/reports/label_distribution.csv
Teacher test: /content/drive/MyDrive/action_classifier/reports/teacher_sanity_test.csv
Summary     : /content/drive/MyDrive/action_classifier/reports/teacher_sanity_summary.json

Manual samples          : 23
Teacher exact accuracy  : 1.0
Teacher avg confidence  : 1.0
Teacher avg latency (s) : 0.799


In [11]:
# =========================================================
# CELL 9 — PUBLIC DATASET SOURCE SETUP
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

PUBLIC_DATASET_NAME = (
    "liontech/NL2Bash"
)

PUBLIC_DATASET_CONFIG = (
    "train"
)

MAX_PUBLIC_SAMPLES = 40000


print()
print("=" * 70)
print("ACTION CLASSIFIER — PUBLIC DATASET SETUP")
print("=" * 70)

print(
    "Dataset source :",
    PUBLIC_DATASET_NAME
)

print(
    "Dataset config :",
    PUBLIC_DATASET_CONFIG
)

print(
    "Max samples    :",
    MAX_PUBLIC_SAMPLES
)

print("=" * 70)


ACTION CLASSIFIER — PUBLIC DATASET SETUP
Dataset source : liontech/NL2Bash
Dataset config : train
Max samples    : 40000


In [12]:
# =========================================================
# CELL 9.1 — DOWNLOAD PUBLIC DATASET
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

from datasets import load_dataset

import os


# =========================================================
# CACHE DIRECTORY
# =========================================================

HF_CACHE_DIR = os.path.join(
    RAW_DIR,
    "hf_cache"
)

os.makedirs(
    HF_CACHE_DIR,
    exist_ok=True
)


# =========================================================
# DOWNLOAD DATASET
# =========================================================

print()
print("=" * 70)
print("ACTION CLASSIFIER — PUBLIC DATASET DOWNLOAD")
print("=" * 70)

print(
    "Dataset:",
    PUBLIC_DATASET_NAME
)

print()


dataset = load_dataset(
    PUBLIC_DATASET_NAME,
    PUBLIC_DATASET_CONFIG,
    cache_dir=HF_CACHE_DIR,
)


# =========================================================
# STATUS
# =========================================================

print()
print("Available splits:")

for split_name in dataset.keys():

    print(
        f"{split_name:12}:",
        len(dataset[split_name])
    )


print()
print(
    "Cache:",
    HF_CACHE_DIR
)

print("=" * 70)


ACTION CLASSIFIER — PUBLIC DATASET DOWNLOAD
Dataset: liontech/NL2Bash



README.md:   0%|          | 0.00/700 [00:00<?, ?B/s]


Available splits:
train       : 40639

Cache: /content/drive/MyDrive/action_classifier/raw/hf_cache


In [13]:
# =========================================================
# CELL 10 — INSPECT PUBLIC DATASET
# MODEL 01 — ACTION CLASSIFIER
# =========================================================


# =========================================================
# SELECT TRAIN SPLIT
# =========================================================

if "train" not in dataset:

    raise RuntimeError(
        "Train split tidak ditemukan."
    )


public_split = dataset[
    "train"
]


# =========================================================
# OPTIONAL LIMIT
# =========================================================

if (
    MAX_PUBLIC_SAMPLES
    and len(public_split) > MAX_PUBLIC_SAMPLES
):

    public_split = (
        public_split
        .shuffle(
            seed=SEED
        )
        .select(
            range(
                MAX_PUBLIC_SAMPLES
            )
        )
    )


# =========================================================
# STATUS
# =========================================================

print()
print("=" * 70)
print("ACTION CLASSIFIER — PUBLIC DATASET INSPECTION")
print("=" * 70)

print(
    "Samples:",
    len(public_split)
)

print(
    "Columns:",
    public_split.column_names
)

print()

print(
    "Features:"
)

print(
    public_split.features
)

print()


for i in range(
    min(
        5,
        len(public_split)
    )
):

    print(
        f"[{i}]",
        public_split[i]
    )

    print()


print("=" * 70)


ACTION CLASSIFIER — PUBLIC DATASET INSPECTION
Samples: 40000
Columns: ['nl', 'bash']

Features:
{'nl': Value('string'), 'bash': Value('string')}

[0] {'nl': 'Convert input_file from one encoding to another and output to stdout', 'bash': 'iconv -f from_encoding -t to_encoding input_file'}

[1] {'nl': 'Find all files with the extension ".txt" in the current directory and its subdirectories up to 3 levels deep, print the results, and replace any numbers in the filenames with a space followed by the number, then sort the results numerically by the number.', 'bash': '`find / -maxdepth 3 -name "*.txt" -print | sed \'s/\\(\\(.*\\)\\([[:digit:]]\\)\\)/\\1 \\3/g\' |sort -n -k 2`'}

[2] {'nl': 'Print the 9th field of all lines beginning with a hyphen (-) in the output of the ls -Rl command.', 'bash': "ls -Rl | awk '/^-/{print $9}'"}

[3] {'nl': 'Find all files with the extension ".js" in the current directory and up to 4 levels of subdirectories, delete them, and then remove all blank lines fro

In [14]:
# =========================================================
# CELL 10 — LOAD + INSPECT PUBLIC DATASET
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

print()
print("=" * 70)
print("LOADING PUBLIC DATASET")
print("=" * 70)

dataset = load_dataset(
    PUBLIC_DATASET_NAME,
    "train"
)

print()
print("Available splits:")

for split_name in dataset.keys():
    print(
        "-",
        split_name,
        ":",
        len(dataset[split_name])
    )


# =========================================================
# SELECT TRAIN SPLIT
# =========================================================

if "train" not in dataset:
    raise RuntimeError(
        "Train split tidak ditemukan."
    )

public_split = dataset["train"]


# =========================================================
# OPTIONAL LIMIT
# =========================================================

if (
    MAX_PUBLIC_SAMPLES
    and len(public_split) > MAX_PUBLIC_SAMPLES
):
    public_split = (
        public_split
        .shuffle(seed=SEED)
        .select(
            range(MAX_PUBLIC_SAMPLES)
        )
    )


# =========================================================
# INSPECTION
# =========================================================

print()
print("=" * 70)
print("ACTION CLASSIFIER — PUBLIC DATASET INSPECTION")
print("=" * 70)

print("Samples:", len(public_split))
print("Columns:", public_split.column_names)

print()
print("Features:")
print(public_split.features)

print()
print("Example records:")

for i in range(
    min(5, len(public_split))
):
    print()
    print(
        f"[{i}]",
        public_split[i]
    )

print()
print("=" * 70)


LOADING PUBLIC DATASET


NL2bash_train.csv:   0%|          | 0.00/5.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/40639 [00:00<?, ? examples/s]


Available splits:
- train : 40639

ACTION CLASSIFIER — PUBLIC DATASET INSPECTION
Samples: 40000
Columns: ['nl', 'bash']

Features:
{'nl': Value('string'), 'bash': Value('string')}

Example records:

[0] {'nl': 'Convert input_file from one encoding to another and output to stdout', 'bash': 'iconv -f from_encoding -t to_encoding input_file'}

[1] {'nl': 'Find all files with the extension ".txt" in the current directory and its subdirectories up to 3 levels deep, print the results, and replace any numbers in the filenames with a space followed by the number, then sort the results numerically by the number.', 'bash': '`find / -maxdepth 3 -name "*.txt" -print | sed \'s/\\(\\(.*\\)\\([[:digit:]]\\)\\)/\\1 \\3/g\' |sort -n -k 2`'}

[2] {'nl': 'Print the 9th field of all lines beginning with a hyphen (-) in the output of the ls -Rl command.', 'bash': "ls -Rl | awk '/^-/{print $9}'"}

[3] {'nl': 'Find all files with the extension ".js" in the current directory and up to 4 levels of subdirector

In [15]:
# =========================================================
# CELL 11 — NORMALIZE PUBLIC DATASET
# MODEL 01 — ACTION CLASSIFIER
# =========================================================


# =========================================================
# 1. COLUMN CANDIDATES
# =========================================================

COMMAND_CANDIDATES = [
    "command",
    "cmd",
    "bash",
    "shell",
    "target",
    "output",
]

DESCRIPTION_CANDIDATES = [
    "description",
    "nl",
    "text",
    "question",
    "instruction",
    "intent",
    "input",
]


available_columns = (
    public_split.column_names
)


print()
print("=" * 70)
print("PUBLIC DATASET — COLUMN DETECTION")
print("=" * 70)

print(
    "Available columns:",
    available_columns
)


# =========================================================
# 2. AUTO-DETECT
# =========================================================

command_column = None

for candidate in COMMAND_CANDIDATES:

    if candidate in available_columns:

        command_column = candidate
        break


description_column = None

for candidate in DESCRIPTION_CANDIDATES:

    if candidate in available_columns:

        description_column = candidate
        break


print()
print(
    "Command column    :",
    command_column
)

print(
    "Description column:",
    description_column
)


if command_column is None:

    raise RuntimeError(
        "Command column tidak berhasil dideteksi. "
        f"Columns: {available_columns}"
    )


# =========================================================
# 3. NORMALIZATION FUNCTIONS
# =========================================================

def normalize_command(text):

    if text is None:
        return ""

    text = str(text)

    text = text.strip()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text


def normalize_description(text):

    if text is None:
        return ""

    text = str(text)

    text = text.strip()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text


# =========================================================
# 4. BUILD DATAFRAME
# =========================================================

records = []


for row in public_split:

    command = normalize_command(
        row.get(
            command_column,
            ""
        )
    )

    if description_column:

        description = normalize_description(
            row.get(
                description_column,
                ""
            )
        )

    else:

        description = ""


    if not command:

        continue


    records.append(
        {
            "command": command,
            "description": description,
            "source": PUBLIC_DATASET_NAME,
            "split_origin": "public_seed",
        }
    )


public_df = pd.DataFrame(
    records
)


print()
print(
    "Normalized samples:",
    len(public_df)
)


print()
display(
    public_df.head(10)
)

print("=" * 70)


PUBLIC DATASET — COLUMN DETECTION
Available columns: ['nl', 'bash']

Command column    : bash
Description column: nl

Normalized samples: 40000



,command,description,source,split_origin
0,iconv -f from_encoding -t to_encoding input_file,Convert input_file from one encoding to anothe...,liontech/NL2Bash,public_seed
1,"`find / -maxdepth 3 -name ""*.txt"" -print | sed...","Find all files with the extension "".txt"" in th...",liontech/NL2Bash,public_seed
2,ls -Rl | awk '/^-/{print $9}',Print the 9th field of all lines beginning wit...,liontech/NL2Bash,public_seed
3,"find / -maxdepth 4 -name ""*.js"" -exec rm -f {}...","Find all files with the extension "".js"" in the...",liontech/NL2Bash,public_seed
4,aws cloudformation detect-stack-drift --stack-...,Start drift detection for the specified CloudF...,liontech/NL2Bash,public_seed
5,find . -name \*\:\*,Find recursively all files under current direc...,liontech/NL2Bash,public_seed
6,"find . -name ""*.txt"" -exec sort -n {} \;","Find all files with the extension "".txt"" and e...",liontech/NL2Bash,public_seed
7,find . -size -300M,find all files in current folder which are les...,liontech/NL2Bash,public_seed
8,xcaddy build --output path/to/file,Build Caddy and output to a specific file,liontech/NL2Bash,public_seed
9,history,show the command history list with line numbers,liontech/NL2Bash,public_seed


In [16]:
# =========================================================
# CELL 12 — EXACT DEDUP + PRELIMINARY DATASET
# MODEL 01 — ACTION CLASSIFIER
# =========================================================


# =========================================================
# 1. PRE-DEDUP STATS
# =========================================================

before_count = len(
    public_df
)


# =========================================================
# 2. DEDUP KEY
# =========================================================

public_df[
    "command_key"
] = (
    public_df["command"]
    .str.strip()
    .str.lower()
)


# =========================================================
# 3. EXACT DUPLICATE CHECK
# =========================================================

duplicate_mask = (
    public_df[
        "command_key"
    ]
    .duplicated(
        keep="first"
    )
)


duplicate_count = int(
    duplicate_mask.sum()
)


duplicates_df = (
    public_df[
        duplicate_mask
    ]
    .copy()
)


dedup_df = (
    public_df[
        ~duplicate_mask
    ]
    .copy()
)


dedup_df = (
    dedup_df
    .drop(
        columns=[
            "command_key"
        ]
    )
    .reset_index(
        drop=True
    )
)


after_count = len(
    dedup_df
)


# =========================================================
# 4. SAVE
# =========================================================

dedup_df.to_json(
    PUBLIC_RAW_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)


dedup_df.to_json(
    DEDUP_DATASET_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)


PUBLIC_DUPLICATES_PATH = (
    f"{REPORT_DIR}/public_exact_duplicates.csv"
)


duplicates_df.to_csv(
    PUBLIC_DUPLICATES_PATH,
    index=False
)


# =========================================================
# 5. BASIC QUALITY STATS
# =========================================================

command_lengths = (
    dedup_df[
        "command"
    ]
    .str.len()
)


description_missing = int(
    (
        dedup_df[
            "description"
        ].str.len()
        == 0
    ).sum()
)


print()
print("=" * 70)
print("ACTION CLASSIFIER — PUBLIC DATASET PRELIMINARY")
print("=" * 70)

print(
    "Before dedup       :",
    before_count
)

print(
    "Exact duplicates   :",
    duplicate_count
)

print(
    "After dedup        :",
    after_count
)

print(
    "Missing description:",
    description_missing
)

print(
    "Avg command length :",
    round(
        command_lengths.mean(),
        2
    )
)

print(
    "Max command length :",
    int(
        command_lengths.max()
    )
)

print()
print(
    "Saved public seed  :",
    PUBLIC_RAW_PATH
)

print(
    "Saved dedup seed   :",
    DEDUP_DATASET_PATH
)

print(
    "Duplicate report   :",
    PUBLIC_DUPLICATES_PATH
)

print("=" * 70)


ACTION CLASSIFIER — PUBLIC DATASET PRELIMINARY
Before dedup       : 40000
Exact duplicates   : 380
After dedup        : 39620
Missing description: 0
Avg command length : 41.94
Max command length : 532

Saved public seed  : /content/drive/MyDrive/action_classifier/raw/public_seed_raw.jsonl
Saved dedup seed   : /content/drive/MyDrive/action_classifier/processed/dedup_dataset_v1.jsonl
Duplicate report   : /content/drive/MyDrive/action_classifier/reports/public_exact_duplicates.csv


In [17]:
# =========================================================
# CELL 13 — QUALITY FILTERING
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

import re


# =========================================================
# FILTER CONFIG
# =========================================================

MIN_COMMAND_LENGTH = 2
MAX_COMMAND_LENGTH = 350

MIN_DESCRIPTION_LENGTH = 3
MAX_DESCRIPTION_LENGTH = 500


# =========================================================
# QUALITY CHECK
# =========================================================

def is_quality_sample(row):

    command = str(
        row["command"]
    ).strip()

    description = str(
        row["description"]
    ).strip()


    if len(command) < MIN_COMMAND_LENGTH:
        return False

    if len(command) > MAX_COMMAND_LENGTH:
        return False

    if len(description) < MIN_DESCRIPTION_LENGTH:
        return False

    if len(description) > MAX_DESCRIPTION_LENGTH:
        return False


    # Reject obvious empty/null strings

    if command.lower() in {
        "none",
        "null",
        "nan",
    }:
        return False


    # Reject commands consisting only
    # of punctuation/whitespace

    if not re.search(
        r"[A-Za-z0-9]",
        command
    ):
        return False


    return True


# =========================================================
# APPLY FILTER
# =========================================================

quality_mask = dedup_df.apply(
    is_quality_sample,
    axis=1
)


quality_df = (
    dedup_df[
        quality_mask
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


rejected_quality_df = (
    dedup_df[
        ~quality_mask
    ]
    .copy()
)


# =========================================================
# SAVE
# =========================================================

QUALITY_DATASET_PATH = (
    f"{PROCESSED_DIR}/quality_filtered_v1.jsonl"
)

QUALITY_REJECTED_PATH = (
    f"{REPORT_DIR}/quality_rejected.csv"
)


quality_df.to_json(
    QUALITY_DATASET_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)


rejected_quality_df.to_csv(
    QUALITY_REJECTED_PATH,
    index=False
)


# =========================================================
# STATUS
# =========================================================

print()
print("=" * 70)
print("ACTION CLASSIFIER — QUALITY FILTER")
print("=" * 70)

print(
    "Input             :",
    len(dedup_df)
)

print(
    "Accepted          :",
    len(quality_df)
)

print(
    "Rejected          :",
    len(rejected_quality_df)
)

print(
    "Acceptance rate   :",
    round(
        len(quality_df)
        /
        len(dedup_df),
        4
    )
)

print()
print(
    "Saved:",
    QUALITY_DATASET_PATH
)

print("=" * 70)


ACTION CLASSIFIER — QUALITY FILTER
Input             : 39620
Accepted          : 39562
Rejected          : 58
Acceptance rate   : 0.9985

Saved: /content/drive/MyDrive/action_classifier/processed/quality_filtered_v1.jsonl


In [18]:
# =========================================================
# CELL 14 — SHELL HEURISTIC ANALYSIS
# MODEL 01 — ACTION CLASSIFIER
# =========================================================


# =========================================================
# HEURISTIC PATTERNS
# =========================================================

HEURISTIC_GROUPS = {

    "READ": [
        r"\bcat\b",
        r"\bls\b",
        r"\bhead\b",
        r"\btail\b",
        r"\bgrep\b",
        r"\bfind\b",
        r"\bstat\b",
        r"\bdu\b",
        r"\bdf\b",
        r"\bpwd\b",
        r"\bwhoami\b",
    ],

    "WRITE": [
        r"\btouch\b",
        r"\bcp\b",
        r"\bmv\b",
        r"\btee\b",
        r">",
        r">>",
    ],

    "DELETE": [
        r"\brm\b",
        r"\brmdir\b",
        r"\bunlink\b",
    ],

    "EXECUTE": [
        r"\bpython(?:3)?\b",
        r"\bbash\b",
        r"\bsh\b",
        r"\bnode\b",
        r"\bperl\b",
        r"\bruby\b",
    ],

    "NETWORK": [
        r"\bcurl\b",
        r"\bwget\b",
        r"\bping\b",
        r"\bssh\b",
        r"\bscp\b",
        r"\brsync\b",
        r"\bnc\b",
        r"\bnetcat\b",
    ],

    "INSTALL": [
        r"\bapt(?:-get)?\b",
        r"\byum\b",
        r"\bdnf\b",
        r"\bpip(?:3)?\b",
        r"\bnpm\b",
        r"\byarn\b",
        r"\bpacman\b",
    ],

    "PRIVILEGED": [
        r"\bsudo\b",
        r"\bsu\b",
    ],

    "SYSTEM_CHANGE": [
        r"\bsystemctl\b",
        r"\bservice\b",
        r"\bchmod\b",
        r"\bchown\b",
        r"\bmount\b",
        r"\bumount\b",
        r"\buseradd\b",
        r"\busermod\b",
        r"\bgroupadd\b",
    ],
}


# =========================================================
# DETECTION
# =========================================================

def detect_heuristics(command):

    command = str(
        command
    ).lower()

    detected = []

    for group, patterns in (
        HEURISTIC_GROUPS.items()
    ):

        for pattern in patterns:

            if re.search(
                pattern,
                command
            ):

                detected.append(
                    group
                )

                break

    return detected


quality_df[
    "heuristic_actions"
] = quality_df[
    "command"
].apply(
    detect_heuristics
)


quality_df[
    "heuristic_count"
] = quality_df[
    "heuristic_actions"
].apply(
    len
)


# =========================================================
# DISTRIBUTION
# =========================================================

heuristic_counts = {
    label: 0
    for label in LABELS
}


for actions in quality_df[
    "heuristic_actions"
]:

    for action in actions:

        heuristic_counts[
            action
        ] += 1


heuristic_distribution_df = (
    pd.DataFrame(
        [
            {
                "label": label,
                "candidate_count": count,
            }

            for label, count
            in heuristic_counts.items()
        ]
    )
)


print()
print("=" * 70)
print("ACTION CLASSIFIER — HEURISTIC COVERAGE")
print("=" * 70)

display(
    heuristic_distribution_df
)


print()

print(
    "No heuristic match :",
    int(
        (
            quality_df[
                "heuristic_count"
            ] == 0
        ).sum()
    )
)

print(
    "1 heuristic match  :",
    int(
        (
            quality_df[
                "heuristic_count"
            ] == 1
        ).sum()
    )
)

print(
    "2+ heuristic match :",
    int(
        (
            quality_df[
                "heuristic_count"
            ] >= 2
        ).sum()
    )
)

print("=" * 70)


ACTION CLASSIFIER — HEURISTIC COVERAGE


,label,candidate_count
0,READ,15643
1,WRITE,2859
2,DELETE,1566
3,EXECUTE,1247
4,NETWORK,861
5,INSTALL,388
6,PRIVILEGED,1202
7,SYSTEM_CHANGE,1794



No heuristic match : 19848
1 heuristic match  : 14249
2+ heuristic match : 5465


In [19]:
# =========================================================
# CELL 15 — BUILD TEACHER CANDIDATE BATCH
# MODEL 01 — ACTION CLASSIFIER
# =========================================================


# =========================================================
# SAMPLING CONFIG
# =========================================================

PER_LABEL_SAMPLE = 250

MULTI_ACTION_SAMPLE = 400

NO_MATCH_SAMPLE = 400

RANDOM_SAMPLE = 500


candidate_parts = []


# =========================================================
# 1. PER-LABEL SAMPLING
# =========================================================

for label in LABELS:

    label_pool = quality_df[
        quality_df[
            "heuristic_actions"
        ].apply(
            lambda x:
                label in x
        )
    ]


    if len(label_pool) == 0:
        continue


    sample_n = min(
        PER_LABEL_SAMPLE,
        len(label_pool)
    )


    sampled = (
        label_pool
        .sample(
            n=sample_n,
            random_state=SEED
        )
        .copy()
    )


    sampled[
        "sampling_reason"
    ] = (
        f"heuristic_{label}"
    )


    candidate_parts.append(
        sampled
    )


# =========================================================
# 2. MULTI-ACTION CANDIDATES
# =========================================================

multi_pool = quality_df[
    quality_df[
        "heuristic_count"
    ] >= 2
]


if len(multi_pool) > 0:

    sample_n = min(
        MULTI_ACTION_SAMPLE,
        len(multi_pool)
    )

    sampled = (
        multi_pool
        .sample(
            n=sample_n,
            random_state=SEED + 1
        )
        .copy()
    )

    sampled[
        "sampling_reason"
    ] = "multi_action"

    candidate_parts.append(
        sampled
    )


# =========================================================
# 3. NO-HEURISTIC MATCH
# =========================================================

no_match_pool = quality_df[
    quality_df[
        "heuristic_count"
    ] == 0
]


if len(no_match_pool) > 0:

    sample_n = min(
        NO_MATCH_SAMPLE,
        len(no_match_pool)
    )

    sampled = (
        no_match_pool
        .sample(
            n=sample_n,
            random_state=SEED + 2
        )
        .copy()
    )

    sampled[
        "sampling_reason"
    ] = "no_heuristic_match"

    candidate_parts.append(
        sampled
    )


# =========================================================
# 4. RANDOM GLOBAL SAMPLE
# =========================================================

sample_n = min(
    RANDOM_SAMPLE,
    len(quality_df)
)


sampled = (
    quality_df
    .sample(
        n=sample_n,
        random_state=SEED + 3
    )
    .copy()
)


sampled[
    "sampling_reason"
] = "random_global"


candidate_parts.append(
    sampled
)


# =========================================================
# MERGE
# =========================================================

teacher_candidate_df = (
    pd.concat(
        candidate_parts,
        ignore_index=True
    )
)


# Remove duplicates created by
# overlapping sampling groups

teacher_candidate_df = (
    teacher_candidate_df
    .drop_duplicates(
        subset=[
            "command"
        ]
    )
    .reset_index(
        drop=True
    )
)


# =========================================================
# SAVE
# =========================================================

TEACHER_CANDIDATE_PATH = (
    f"{TEACHER_DIR}/teacher_candidates_v1.jsonl"
)


teacher_candidate_df.to_json(
    TEACHER_CANDIDATE_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)


# =========================================================
# STATUS
# =========================================================

print()
print("=" * 70)
print("ACTION CLASSIFIER — TEACHER CANDIDATES")
print("=" * 70)

print(
    "Candidates:",
    len(teacher_candidate_df)
)

print()

print(
    teacher_candidate_df[
        "sampling_reason"
    ].value_counts()
)

print()

print(
    "Saved:",
    TEACHER_CANDIDATE_PATH
)

print("=" * 70)


ACTION CLASSIFIER — TEACHER CANDIDATES
Candidates: 3174

sampling_reason
random_global              477
no_heuristic_match         400
multi_action               340
heuristic_READ             250
heuristic_WRITE            248
heuristic_INSTALL          248
heuristic_DELETE           246
heuristic_NETWORK          246
heuristic_EXECUTE          242
heuristic_PRIVILEGED       240
heuristic_SYSTEM_CHANGE    237
Name: count, dtype: int64

Saved: /content/drive/MyDrive/action_classifier/teacher/teacher_candidates_v1.jsonl


In [20]:
# =========================================================
# CELL 16 — TEACHER LABELING DRY RUN
# MODEL 01 — ACTION CLASSIFIER
# =========================================================


# =========================================================
# CONFIG
# =========================================================

DRY_RUN_SIZE = 40

DRY_RUN_DELAY = 0.15


# =========================================================
# SAMPLE
# =========================================================

dry_run_df = (
    teacher_candidate_df
    .sample(
        n=min(
            DRY_RUN_SIZE,
            len(
                teacher_candidate_df
            )
        ),
        random_state=SEED
    )
    .copy()
    .reset_index(
        drop=True
    )
)


results = []


print()
print("=" * 90)
print("ACTION CLASSIFIER — TEACHER DRY RUN")
print("=" * 90)


for index, row in dry_run_df.iterrows():

    command = row[
        "command"
    ]

    print()
    print(
        f"[{index + 1}/{len(dry_run_df)}]",
        command[:140]
    )


    try:

        raw_output, latency = (
            call_teacher(
                command
            )
        )


        parsed = (
            parse_teacher_output(
                raw_output
            )
        )


        result = {

            "command":
                command,

            "description":
                row[
                    "description"
                ],

            "actions":
                parsed[
                    "actions"
                ],

            "confidence":
                parsed[
                    "confidence"
                ],

            "ambiguous":
                parsed[
                    "ambiguous"
                ],

            "latency":
                latency,

            "sampling_reason":
                row[
                    "sampling_reason"
                ],

            "error":
                None,
        }


        print(
            "Actions   :",
            parsed[
                "actions"
            ]
        )

        print(
            "Confidence:",
            parsed[
                "confidence"
            ]
        )


    except Exception as e:

        result = {

            "command":
                command,

            "description":
                row[
                    "description"
                ],

            "actions":
                None,

            "confidence":
                None,

            "ambiguous":
                None,

            "latency":
                None,

            "sampling_reason":
                row[
                    "sampling_reason"
                ],

            "error":
                str(e),
        }


        print(
            "ERROR:",
            str(e)
        )


    results.append(
        result
    )


    time.sleep(
        DRY_RUN_DELAY
    )


# =========================================================
# RESULTS
# =========================================================

teacher_dry_df = (
    pd.DataFrame(
        results
    )
)


successful = int(
    teacher_dry_df[
        "error"
    ].isna().sum()
)


ambiguous_count = int(
    teacher_dry_df[
        "ambiguous"
    ].fillna(False).sum()
)


average_confidence = (
    teacher_dry_df[
        "confidence"
    ]
    .dropna()
    .mean()
)


average_latency = (
    teacher_dry_df[
        "latency"
    ]
    .dropna()
    .mean()
)


# =========================================================
# SAVE
# =========================================================

TEACHER_DRY_RUN_PATH = (
    f"{REPORT_DIR}/teacher_dry_run.csv"
)


save_df = (
    teacher_dry_df.copy()
)


save_df[
    "actions"
] = save_df[
    "actions"
].apply(
    lambda x:
        json.dumps(x)
        if isinstance(x, list)
        else x
)


save_df.to_csv(
    TEACHER_DRY_RUN_PATH,
    index=False
)


# =========================================================
# STATUS
# =========================================================

print()
print("=" * 90)
print("TEACHER DRY RUN SUMMARY")
print("=" * 90)

print(
    "Total       :",
    len(
        teacher_dry_df
    )
)

print(
    "Successful  :",
    successful
)

print(
    "Errors      :",
    len(
        teacher_dry_df
    ) - successful
)

print(
    "Ambiguous   :",
    ambiguous_count
)

print(
    "Avg conf    :",
    round(
        average_confidence,
        4
    )
)

print(
    "Avg latency :",
    round(
        average_latency,
        3
    ),
    "sec"
)

print()
print(
    "Saved:",
    TEACHER_DRY_RUN_PATH
)

print("=" * 90)


ACTION CLASSIFIER — TEACHER DRY RUN

[1/40] find ~ -iname '*.jpg' -mtime -1 | xargs -i rm -f {}
Actions   : ['READ', 'DELETE']
Confidence: 0.95

[2/40] find ~/ -exec touch {} \;
Actions   : ['WRITE', 'EXECUTE']
Confidence: 0.95

[3/40] uvicorn --reload import.path:app_object
Actions   : ['EXECUTE']
Confidence: 0.95

[4/40] sudo synoupgrade --start
Actions   : ['EXECUTE', 'PRIVILEGED', 'SYSTEM_CHANGE']
Confidence: 0.95

[5/40] ssh -fNT -L8888:proxyhost:8888 -R22222:localhost:22 officefirewall
Actions   : ['EXECUTE', 'NETWORK', 'SYSTEM_CHANGE']
Confidence: 0.95

[6/40] bchunk -v path/to/image.bin path/to/image.cue path/to/output
Actions   : ['READ', 'WRITE', 'EXECUTE']
Confidence: 0.95

[7/40] find -type f -exec chmod -v +x {} \;
Actions   : ['READ', 'SYSTEM_CHANGE']
Confidence: 0.95

[8/40] msfvenom -l formats
Actions   : ['READ']
Confidence: 1.0

[9/40] echo $(shuf -n 1 -e $(ls -p ~/ | grep -v / | tr "\n" " "))
Actions   : ['READ', 'EXECUTE']
Confidence: 0.9

[10/40] sudo iotop --only

In [21]:
# =========================================================
# CELL 16.1 — TEACHER AUDIT EXPORT
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

AUDIT_SIZE = 25


# =========================================================
# SELECT AUDIT SAMPLES
# =========================================================

teacher_audit_df = (
    teacher_dry_df[
        teacher_dry_df["error"].isna()
    ]
    .sample(
        n=min(
            AUDIT_SIZE,
            len(
                teacher_dry_df[
                    teacher_dry_df["error"].isna()
                ]
            )
        ),
        random_state=SEED
    )
    .copy()
    .reset_index(
        drop=True
    )
)


# =========================================================
# PREPARE MANUAL LABEL COLUMN
# =========================================================

teacher_audit_df[
    "teacher_actions"
] = teacher_audit_df[
    "actions"
]


teacher_audit_df[
    "manual_actions"
] = None


teacher_audit_df[
    "manual_notes"
] = ""


# =========================================================
# DISPLAY
# =========================================================

print()
print("=" * 90)
print("ACTION CLASSIFIER — TEACHER AUDIT SET")
print("=" * 90)

for i, row in teacher_audit_df.iterrows():

    print()
    print(
        f"[{i}]"
    )

    print(
        "COMMAND :",
        row["command"]
    )

    print(
        "TEACHER :",
        row["teacher_actions"]
    )

    print(
        "CONF    :",
        row["confidence"]
    )

    print("-" * 90)


# =========================================================
# SAVE CSV
# =========================================================

TEACHER_AUDIT_PATH = (
    f"{REPORT_DIR}/teacher_audit_manual.csv"
)


save_audit_df = (
    teacher_audit_df.copy()
)


save_audit_df[
    "teacher_actions"
] = save_audit_df[
    "teacher_actions"
].apply(
    json.dumps
)


save_audit_df[
    "actions"
] = save_audit_df[
    "actions"
].apply(
    json.dumps
)


save_audit_df.to_csv(
    TEACHER_AUDIT_PATH,
    index=False
)


print()
print(
    "Audit file:",
    TEACHER_AUDIT_PATH
)

print("=" * 90)


ACTION CLASSIFIER — TEACHER AUDIT SET

[0]
COMMAND : find /home -type f -name *.mp4 -size +10M -exec rm {} \;
TEACHER : ['READ', 'DELETE']
CONF    : 0.95
------------------------------------------------------------------------------------------

[1]
COMMAND : find /usr/local -type d -empty -exec rm -rvf {} \;
TEACHER : ['READ', 'DELETE']
CONF    : 0.95
------------------------------------------------------------------------------------------

[2]
COMMAND : yes | cp $(date +%s) /dev/null
TEACHER : ['READ', 'WRITE', 'EXECUTE']
CONF    : 0.9
------------------------------------------------------------------------------------------

[3]
COMMAND : find / -iname *.txt | xargs head -n 5
TEACHER : ['READ']
CONF    : 0.95
------------------------------------------------------------------------------------------

[4]
COMMAND : ssh -fNT -L8888:proxyhost:8888 -R22222:localhost:22 officefirewall
TEACHER : ['EXECUTE', 'NETWORK', 'SYSTEM_CHANGE']
CONF    : 0.95
--------------------------------------

In [22]:
# =========================================================
# CELL 16.1B — SHOW AUDIT SAMPLES FOR MANUAL LABELING
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

print()
print("=" * 90)
print("MANUAL LABELING — AUDIT SAMPLES")
print("=" * 90)

for i, row in teacher_audit_df.iterrows():

    print()
    print(f"[{i}]")
    print("COMMAND :", row["command"])
    print("TEACHER :", row["teacher_actions"])
    print("CONF    :", row["confidence"])
    print("-" * 90)


MANUAL LABELING — AUDIT SAMPLES

[0]
COMMAND : find /home -type f -name *.mp4 -size +10M -exec rm {} \;
TEACHER : ['READ', 'DELETE']
CONF    : 0.95
------------------------------------------------------------------------------------------

[1]
COMMAND : find /usr/local -type d -empty -exec rm -rvf {} \;
TEACHER : ['READ', 'DELETE']
CONF    : 0.95
------------------------------------------------------------------------------------------

[2]
COMMAND : yes | cp $(date +%s) /dev/null
TEACHER : ['READ', 'WRITE', 'EXECUTE']
CONF    : 0.9
------------------------------------------------------------------------------------------

[3]
COMMAND : find / -iname *.txt | xargs head -n 5
TEACHER : ['READ']
CONF    : 0.95
------------------------------------------------------------------------------------------

[4]
COMMAND : ssh -fNT -L8888:proxyhost:8888 -R22222:localhost:22 officefirewall
TEACHER : ['EXECUTE', 'NETWORK', 'SYSTEM_CHANGE']
CONF    : 0.95
--------------------------------------------

In [23]:
# =========================================================
# CELL 16.2 — TEACHER VS MANUAL COMPARISON
# SAFE VERSION
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

import json
import pandas as pd


# =========================================================
# NORMALIZER
# =========================================================

def normalize_actions_safe(value):

    if value is None:
        return set()

    if isinstance(value, float) and pd.isna(value):
        return set()

    if isinstance(value, str):

        value = value.strip()

        if not value:
            return set()

        try:
            value = json.loads(value)

        except Exception:

            value = [
                x.strip()
                for x in value.split(",")
                if x.strip()
            ]


    if isinstance(value, (list, tuple, set)):

        return {
            str(x).strip().upper()
            for x in value
            if str(x).strip()
        }


    return set()


# =========================================================
# COPY AUDIT DATA
# =========================================================

audit_compare_df = teacher_audit_df.copy()


print()
print("=" * 90)
print("TEACHER VS MANUAL AUDIT")
print("=" * 90)

print(
    "Rows before filter:",
    len(audit_compare_df)
)


# =========================================================
# CHECK REQUIRED COLUMNS
# =========================================================

required_columns = [
    "teacher_actions",
    "manual_actions",
]

missing_columns = [
    col
    for col in required_columns
    if col not in audit_compare_df.columns
]


if missing_columns:

    raise RuntimeError(
        "Kolom tidak ditemukan: "
        + str(missing_columns)
    )


# =========================================================
# NORMALIZE INTO NEW COLUMNS
# =========================================================

teacher_sets = []
manual_sets = []


for teacher_value, manual_value in zip(
    audit_compare_df["teacher_actions"],
    audit_compare_df["manual_actions"]
):

    teacher_sets.append(
        normalize_actions_safe(
            teacher_value
        )
    )

    manual_sets.append(
        normalize_actions_safe(
            manual_value
        )
    )


audit_compare_df[
    "_teacher_set"
] = teacher_sets

audit_compare_df[
    "_manual_set"
] = manual_sets


# =========================================================
# KEEP ONLY MANUALLY LABELED ROWS
# =========================================================

manual_mask = [
    len(x) > 0
    for x in audit_compare_df[
        "_manual_set"
    ]
]


audit_compare_df = (
    audit_compare_df[
        manual_mask
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print(
    "Rows with manual label:",
    len(audit_compare_df)
)


# =========================================================
# NO LABELS YET
# =========================================================

if len(audit_compare_df) == 0:

    print()
    print(
        "BELUM ADA MANUAL GROUND TRUTH."
    )

    print(
        "manual_actions masih kosong."
    )

    print()
    print(
        "Teacher belum bisa dibandingkan "
        "sebelum label manual diisi."
    )

    print("=" * 90)


# =========================================================
# COMPARE
# =========================================================

else:

    exact_results = []

    for i in range(
        len(audit_compare_df)
    ):

        teacher_set = (
            audit_compare_df
            .at[
                i,
                "_teacher_set"
            ]
        )

        manual_set = (
            audit_compare_df
            .at[
                i,
                "_manual_set"
            ]
        )

        exact_results.append(
            teacher_set == manual_set
        )


    audit_compare_df[
        "exact_match"
    ] = pd.Series(
        exact_results,
        dtype=bool
    )


    # =====================================================
    # SUMMARY
    # =====================================================

    total = len(
        audit_compare_df
    )

    matches = int(
        audit_compare_df[
            "exact_match"
        ].sum()
    )

    accuracy = (
        matches / total
    )


    print()
    print(
        "Samples       :",
        total
    )

    print(
        "Exact matches :",
        matches
    )

    print(
        "Exact accuracy:",
        round(
            accuracy,
            4
        )
    )


    # =====================================================
    # MISMATCHES
    # =====================================================

    mismatch_df = (
        audit_compare_df[
            audit_compare_df[
                "exact_match"
            ] == False
        ]
        .copy()
    )


    print(
        "Mismatches    :",
        len(
            mismatch_df
        )
    )


    if len(mismatch_df) > 0:

        display(
            mismatch_df[
                [
                    "command",
                    "teacher_actions",
                    "manual_actions",
                    "confidence",
                ]
            ]
        )


    print("=" * 90)


TEACHER VS MANUAL AUDIT
Rows before filter: 25
Rows with manual label: 0

BELUM ADA MANUAL GROUND TRUTH.
manual_actions masih kosong.

Teacher belum bisa dibandingkan sebelum label manual diisi.


In [24]:
# =========================================================
# CELL 16.1C — MANUAL GROUND TRUTH
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

MANUAL_AUDIT_LABELS = {

    0: [
        "READ",
        "DELETE",
    ],

    1: [
        "READ",
        "DELETE",
    ],

    2: [
        "READ",
        "WRITE",
        "EXECUTE",
    ],

    3: [
        "READ",
    ],

    4: [
        "NETWORK",
        "EXECUTE",
    ],

    5: [
        "NETWORK",
        "PRIVILEGED",
    ],

    6: [
        "PRIVILEGED",
        "SYSTEM_CHANGE",
    ],

    7: [
        "PRIVILEGED",
        "SYSTEM_CHANGE",
    ],

    8: [
        "READ",
        "DELETE",
    ],

    9: [
        "READ",
        "SYSTEM_CHANGE",
    ],

    10: [
        "READ",
        "NETWORK",
    ],

    11: [
        "READ",
        "PRIVILEGED",
    ],

    12: [
        "READ",
        "WRITE",
    ],

    13: [
        "READ",
        "NETWORK",
    ],

    14: [
        "NETWORK",
        "WRITE",
        "INSTALL",
        "SYSTEM_CHANGE",
    ],

    15: [
        "READ",
        "EXECUTE",
    ],

    16: [
        "READ",
        "WRITE",
    ],

    17: [
        "READ",
        "WRITE",
    ],

    18: [
        "READ",
        "DELETE",
    ],

    19: [
        "NETWORK",
        "WRITE",
        "PRIVILEGED",
    ],

    20: [
        "READ",
        "WRITE",
    ],

    21: [
        "SYSTEM_CHANGE",
    ],

    22: [
        "READ",
        "WRITE",
    ],

    23: [
        "READ",
    ],

    24: [
        "READ",
        "WRITE",
    ],
}


# =========================================================
# APPLY MANUAL LABELS
# =========================================================

for index, labels in MANUAL_AUDIT_LABELS.items():

    teacher_audit_df.at[
        index,
        "manual_actions"
    ] = labels


# =========================================================
# VALIDATE
# =========================================================

filled = teacher_audit_df[
    "manual_actions"
].notna().sum()


print()
print("=" * 70)
print("MANUAL GROUND TRUTH STATUS")
print("=" * 70)

print(
    "Filled:",
    filled,
    "/",
    len(teacher_audit_df)
)

print("=" * 70)


assert filled == len(
    teacher_audit_df
), "Masih ada manual label yang kosong."


MANUAL GROUND TRUTH STATUS
Filled: 25 / 25


In [25]:
# =========================================================
# CELL 17 — MASS TEACHER LABELING CONFIG
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

import os
import json
import time
import pandas as pd


# =========================================================
# CONFIG
# =========================================================

TEACHER_BATCH_LIMIT = None
# None = label semua candidate
# contoh 500 = hanya 500 sample pertama

REQUEST_DELAY = 0.10

MAX_RETRIES = 3

RETRY_DELAY = 2.0

CHECKPOINT_EVERY = 25


# =========================================================
# OUTPUT PATHS
# =========================================================

TEACHER_PROGRESS_PATH = (
    f"{TEACHER_DIR}/teacher_label_progress_v1.jsonl"
)

TEACHER_ERROR_PATH = (
    f"{TEACHER_DIR}/teacher_label_errors_v1.jsonl"
)


# =========================================================
# LOAD CANDIDATES
# =========================================================

mass_candidate_df = (
    teacher_candidate_df.copy()
)


if TEACHER_BATCH_LIMIT is not None:

    mass_candidate_df = (
        mass_candidate_df
        .head(
            TEACHER_BATCH_LIMIT
        )
        .copy()
    )


# =========================================================
# LOAD COMPLETED COMMANDS
# =========================================================

completed_commands = set()


if os.path.exists(
    TEACHER_PROGRESS_PATH
):

    previous_df = pd.read_json(
        TEACHER_PROGRESS_PATH,
        lines=True
    )

    if "command" in previous_df.columns:

        completed_commands = set(
            previous_df[
                "command"
            ].astype(str)
        )


remaining_df = (
    mass_candidate_df[
        ~mass_candidate_df[
            "command"
        ].astype(str).isin(
            completed_commands
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# =========================================================
# STATUS
# =========================================================

print()
print("=" * 80)
print("ACTION CLASSIFIER — MASS LABELING SETUP")
print("=" * 80)

print(
    "Total candidates :",
    len(mass_candidate_df)
)

print(
    "Already labeled  :",
    len(completed_commands)
)

print(
    "Remaining        :",
    len(remaining_df)
)

print(
    "Checkpoint every :",
    CHECKPOINT_EVERY
)

print(
    "Progress file    :",
    TEACHER_PROGRESS_PATH
)

print("=" * 80)


ACTION CLASSIFIER — MASS LABELING SETUP
Total candidates : 3174
Already labeled  : 3171
Remaining        : 3
Checkpoint every : 25
Progress file    : /content/drive/MyDrive/action_classifier/teacher/teacher_label_progress_v1.jsonl


In [26]:
# =========================================================
# CELL 18 — MASS TEACHER LABELING
# RESUMABLE + CHECKPOINT SAFE VERSION
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

import os
import json
import time
import pandas as pd


# =========================================================
# SAFE JSONL APPEND
# =========================================================

def append_jsonl(
    path,
    record
):

    line = (
        json.dumps(
            record,
            ensure_ascii=False
        )
        + "\n"
    )

    with open(
        path,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            line
        )

        f.flush()

        os.fsync(
            f.fileno()
        )


# =========================================================
# SAFE JSONL LOADER
#
# Ignores corrupted/incomplete final line.
# =========================================================

def load_jsonl_safe(
    path
):

    records = []

    if not os.path.exists(
        path
    ):

        return records


    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line_number, line in enumerate(
            f,
            start=1
        ):

            line = line.strip()

            if not line:
                continue

            try:

                records.append(
                    json.loads(
                        line
                    )
                )

            except json.JSONDecodeError:

                print(
                    f"WARNING: corrupted JSONL "
                    f"line skipped: {line_number}"
                )


    return records


# =========================================================
# TEACHER WITH RETRY
# =========================================================

def call_teacher_with_retry(
    command,
    max_retries=MAX_RETRIES
):

    last_error = None


    for attempt in range(
        1,
        max_retries + 1
    ):

        try:

            raw_output, latency = (
                call_teacher(
                    command
                )
            )

            parsed = (
                parse_teacher_output(
                    raw_output
                )
            )


            return (
                parsed,
                latency,
                None
            )


        except Exception as e:

            last_error = str(e)


            print(
                f"Retry "
                f"{attempt}/{max_retries}:",
                last_error[:160]
            )


            time.sleep(
                RETRY_DELAY
                * attempt
            )


    return (
        None,
        None,
        last_error
    )


# =========================================================
# READ CURRENT CHECKPOINT
# =========================================================

existing_records = (
    load_jsonl_safe(
        TEACHER_PROGRESS_PATH
    )
)


already_saved = {
    str(
        record.get(
            "command",
            ""
        )
    )

    for record
    in existing_records

    if record.get(
        "command"
    )
}


print()
print("=" * 90)
print("ACTION CLASSIFIER — MASS TEACHER LABELING")
print("=" * 90)

print(
    "Existing checkpoint:",
    len(
        already_saved
    )
)

print(
    "Current batch       :",
    len(
        remaining_df
    )
)

print("=" * 90)


# =========================================================
# COUNTERS
# =========================================================

success_count = 0

error_count = 0

skipped_count = 0


# =========================================================
# LABEL LOOP
# =========================================================

for index, row in remaining_df.iterrows():

    command = str(
        row[
            "command"
        ]
    ).strip()


    # =====================================================
    # SKIP IF ALREADY SAVED
    # =====================================================

    if command in already_saved:

        skipped_count += 1

        continue


    parsed, latency, error = (
        call_teacher_with_retry(
            command
        )
    )


    # =====================================================
    # SUCCESS
    # =====================================================

    if error is None:

        record = {

            "command":
                command,

            "description":
                row.get(
                    "description",
                    ""
                ),

            "actions":
                parsed[
                    "actions"
                ],

            "confidence":
                parsed[
                    "confidence"
                ],

            "ambiguous":
                parsed[
                    "ambiguous"
                ],

            "latency":
                latency,

            "source":
                row.get(
                    "source",
                    PUBLIC_DATASET_NAME
                ),

            "split_origin":
                row.get(
                    "split_origin",
                    "public_seed"
                ),

            "sampling_reason":
                row.get(
                    "sampling_reason",
                    ""
                ),

            "teacher_model":
                MODEL,

            "timestamp":
                time.time(),
        }


        append_jsonl(
            TEACHER_PROGRESS_PATH,
            record
        )


        # Keep in memory too
        already_saved.add(
            command
        )


        success_count += 1


    # =====================================================
    # ERROR
    # =====================================================

    else:

        error_record = {

            "command":
                command,

            "error":
                error,

            "teacher_model":
                MODEL,

            "timestamp":
                time.time(),
        }


        append_jsonl(
            TEACHER_ERROR_PATH,
            error_record
        )


        error_count += 1


    # =====================================================
    # CHECKPOINT STATUS
    # =====================================================

    processed_now = (
        success_count
        + error_count
    )


    if (
        processed_now > 0
        and (
            processed_now
            % CHECKPOINT_EVERY
            == 0
        )
    ):

        total_saved = len(
            already_saved
        )


        print()
        print(
            f"[CHECKPOINT] "
            f"new={processed_now} "
            f"success={success_count} "
            f"errors={error_count} "
            f"total_saved={total_saved}"
        )


    time.sleep(
        REQUEST_DELAY
    )


# =========================================================
# FINAL VALIDATION
# =========================================================

final_records = (
    load_jsonl_safe(
        TEACHER_PROGRESS_PATH
    )
)


final_commands = {
    str(
        record.get(
            "command",
            ""
        )
    )

    for record
    in final_records

    if record.get(
        "command"
    )
}


# =========================================================
# STATUS
# =========================================================

print()
print("=" * 90)
print("MASS LABELING SESSION COMPLETE")
print("=" * 90)

print(
    "New success :",
    success_count
)

print(
    "New errors  :",
    error_count
)

print(
    "Skipped     :",
    skipped_count
)

print(
    "Total saved :",
    len(
        final_commands
    )
)

print(
    "Progress    :",
    TEACHER_PROGRESS_PATH
)

print(
    "Errors      :",
    TEACHER_ERROR_PATH
)

print("=" * 90)


ACTION CLASSIFIER — MASS TEACHER LABELING
Existing checkpoint: 3171
Current batch       : 3
Retry 1/3: actions kosong.
Retry 2/3: actions kosong.
Retry 3/3: actions kosong.
Retry 1/3: actions kosong.
Retry 2/3: actions kosong.
Retry 3/3: actions kosong.
Retry 1/3: actions kosong.
Retry 2/3: actions kosong.
Retry 3/3: actions kosong.

MASS LABELING SESSION COMPLETE
New success : 0
New errors  : 3
Skipped     : 0
Total saved : 3171
Progress    : /content/drive/MyDrive/action_classifier/teacher/teacher_label_progress_v1.jsonl
Errors      : /content/drive/MyDrive/action_classifier/teacher/teacher_label_errors_v1.jsonl


In [27]:
# =========================================================
# CELL 19 — TEACHER RESULT FILTERING
# MODEL 01 — ACTION CLASSIFIER
# =========================================================


# =========================================================
# FILTER CONFIG
# =========================================================

MIN_TEACHER_CONFIDENCE = 0.90


# =========================================================
# LOAD RESULTS
# =========================================================

teacher_all_df = pd.read_json(
    TEACHER_PROGRESS_PATH,
    lines=True
)


# Safety dedup

teacher_all_df = (
    teacher_all_df
    .drop_duplicates(
        subset=[
            "command"
        ],
        keep="last"
    )
    .reset_index(
        drop=True
    )
)


# =========================================================
# VALIDATE ACTION LABELS
# =========================================================

teacher_all_df[
    "valid_actions"
] = teacher_all_df[
    "actions"
].apply(
    validate_actions
)


# =========================================================
# ACCEPTANCE MASK
# =========================================================

trusted_mask = (

    teacher_all_df[
        "valid_actions"
    ]

    &

    (
        teacher_all_df[
            "confidence"
        ] >= MIN_TEACHER_CONFIDENCE
    )

    &

    (
        teacher_all_df[
            "ambiguous"
        ] == False
    )
)


teacher_trusted_df = (
    teacher_all_df[
        trusted_mask
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


teacher_rejected_df = (
    teacher_all_df[
        ~trusted_mask
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# =========================================================
# SAVE
# =========================================================

teacher_trusted_df.to_json(
    TEACHER_LABELED_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)


teacher_rejected_df.to_json(
    TEACHER_REJECTED_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)


# =========================================================
# STATUS
# =========================================================

print()
print("=" * 80)
print("ACTION CLASSIFIER — TEACHER FILTERING")
print("=" * 80)

print(
    "Teacher results :",
    len(teacher_all_df)
)

print(
    "Accepted        :",
    len(teacher_trusted_df)
)

print(
    "Rejected        :",
    len(teacher_rejected_df)
)

print(
    "Acceptance rate :",
    round(
        len(teacher_trusted_df)
        /
        max(
            len(teacher_all_df),
            1
        ),
        4
    )
)

print()

print(
    "Mean confidence :",
    round(
        teacher_all_df[
            "confidence"
        ].mean(),
        4
    )
)

print(
    "Min confidence  :",
    round(
        teacher_all_df[
            "confidence"
        ].min(),
        4
    )
)

print("=" * 80)


ACTION CLASSIFIER — TEACHER FILTERING
Teacher results : 3171
Accepted        : 3145
Rejected        : 26
Acceptance rate : 0.9918

Mean confidence : 0.9446
Min confidence  : 0.6


In [29]:
# =========================================================
# CELL 20 — LABEL DISTRIBUTION AUDIT
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

from collections import Counter


# =========================================================
# LABEL COUNTS
# =========================================================

label_counter = Counter()


for actions in teacher_trusted_df[
    "actions"
]:

    for action in actions:

        label_counter[
            action
        ] += 1


label_distribution = []


for label in LABELS:

    count = (
        label_counter[
            label
        ]
    )

    percentage = (
        count
        /
        max(
            len(
                teacher_trusted_df
            ),
            1
        )
    )


    label_distribution.append(
        {
            "label":
                label,

            "count":
                count,

            "sample_ratio":
                percentage,
        }
    )


label_distribution_df = (
    pd.DataFrame(
        label_distribution
    )
)


# =========================================================
# ACTIONS PER SAMPLE
# =========================================================

teacher_trusted_df[
    "action_count"
] = teacher_trusted_df[
    "actions"
].apply(
    len
)


action_count_distribution = (
    teacher_trusted_df[
        "action_count"
    ]
    .value_counts()
    .sort_index()
)


# =========================================================
# LABEL COMBINATIONS
# =========================================================

teacher_trusted_df[
    "label_combo"
] = teacher_trusted_df[
    "actions"
].apply(
    lambda x:
        " + ".join(
            sorted(x)
        )
)


combo_distribution = (
    teacher_trusted_df[
        "label_combo"
    ]
    .value_counts()
    .head(20)
)


# =========================================================
# SAVE
# =========================================================

label_distribution_df.to_csv(
    LABEL_DISTRIBUTION_PATH,
    index=False
)


LABEL_COMBO_PATH = (
    f"{REPORT_DIR}/label_combinations.csv"
)


teacher_trusted_df[
    "label_combo"
].value_counts().rename_axis(
    "combination"
).reset_index(
    name="count"
).to_csv(
    LABEL_COMBO_PATH,
    index=False
)


# =========================================================
# DISPLAY
# =========================================================

print()
print("=" * 80)
print("ACTION CLASSIFIER — LABEL DISTRIBUTION")
print("=" * 80)

display(
    label_distribution_df
)


print()
print("ACTIONS PER SAMPLE")
print("-" * 80)

print(
    action_count_distribution
)


print()
print("TOP LABEL COMBINATIONS")
print("-" * 80)

print(
    combo_distribution
)


print()
print(
    "Total trusted:",
    len(
        teacher_trusted_df
    )
)

print("=" * 80)


ACTION CLASSIFIER — LABEL DISTRIBUTION


,label,count,sample_ratio
0,READ,2153,0.684579
1,WRITE,816,0.259459
2,DELETE,422,0.134181
3,EXECUTE,949,0.301749
4,NETWORK,458,0.145628
5,INSTALL,73,0.023211
6,PRIVILEGED,341,0.108426
7,SYSTEM_CHANGE,845,0.268680



ACTIONS PER SAMPLE
--------------------------------------------------------------------------------
action_count
1     981
2    1505
3     576
4      77
5       6
Name: count, dtype: int64

TOP LABEL COMBINATIONS
--------------------------------------------------------------------------------
label_combo
READ                                    591
READ + SYSTEM_CHANGE                    297
READ + WRITE                            264
DELETE + READ                           262
EXECUTE                                 182
EXECUTE + READ + WRITE                  181
EXECUTE + READ                          162
NETWORK + READ                           95
PRIVILEGED + SYSTEM_CHANGE               87
EXECUTE + NETWORK                        80
READ + SYSTEM_CHANGE + WRITE             71
SYSTEM_CHANGE                            71
EXECUTE + PRIVILEGED + SYSTEM_CHANGE     65
DELETE                                   60
EXECUTE + WRITE                          53
DELETE + EXECUTE + READ          

In [30]:
# =========================================================
# CELL 21 — FREEZE TRUSTED DATASET V1
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

import pandas as pd
import json


# =========================================================
# 1. PREPARE TEACHER TRUSTED
# =========================================================

teacher_freeze_df = (
    teacher_trusted_df.copy()
)


# =========================================================
# 2. PREPARE MANUAL SEED
# =========================================================

manual_freeze_df = (
    manual_df.copy()
)


# Ensure same columns

for column in teacher_freeze_df.columns:

    if column not in manual_freeze_df.columns:

        manual_freeze_df[
            column
        ] = None


# =========================================================
# 3. MERGE
# =========================================================

trusted_v1_df = pd.concat(
    [
        manual_freeze_df,
        teacher_freeze_df,
    ],
    ignore_index=True
)


# =========================================================
# 4. FINAL CLEANUP
# =========================================================

trusted_v1_df[
    "command"
] = (
    trusted_v1_df[
        "command"
    ]
    .astype(str)
    .str.strip()
)


trusted_v1_df = (
    trusted_v1_df[
        trusted_v1_df[
            "command"
        ].str.len() > 0
    ]
    .copy()
)


# Manual data wins on duplicate command

trusted_v1_df[
    "_manual_priority"
] = (
    trusted_v1_df[
        "source"
    ].eq(
        "manual_seed"
    )
    .astype(int)
)


trusted_v1_df = (
    trusted_v1_df
    .sort_values(
        "_manual_priority",
        ascending=False
    )
    .drop_duplicates(
        subset=[
            "command"
        ],
        keep="first"
    )
    .drop(
        columns=[
            "_manual_priority"
        ]
    )
    .reset_index(
        drop=True
    )
)


# =========================================================
# 5. VALIDATE LABELS
# =========================================================

trusted_v1_df[
    "_valid"
] = trusted_v1_df[
    "actions"
].apply(
    validate_actions
)


invalid_rows = (
    trusted_v1_df[
        ~trusted_v1_df[
            "_valid"
        ]
    ]
)


assert (
    len(invalid_rows) == 0
), (
    f"Invalid label rows: {len(invalid_rows)}"
)


trusted_v1_df = (
    trusted_v1_df
    .drop(
        columns=[
            "_valid"
        ]
    )
)


# =========================================================
# 6. SAVE
# =========================================================

trusted_v1_df.to_json(
    TRUSTED_TRAIN_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)


# =========================================================
# 7. SUMMARY
# =========================================================

print()
print("=" * 80)
print("ACTION CLASSIFIER — TRUSTED DATASET V1")
print("=" * 80)

print(
    "Manual seed     :",
    len(
        manual_freeze_df
    )
)

print(
    "Teacher trusted :",
    len(
        teacher_freeze_df
    )
)

print(
    "Final trusted   :",
    len(
        trusted_v1_df
    )
)

print(
    "Saved           :",
    TRUSTED_TRAIN_PATH
)

print("=" * 80)


ACTION CLASSIFIER — TRUSTED DATASET V1
Manual seed     : 23
Teacher trusted : 3145
Final trusted   : 3167
Saved           : /content/drive/MyDrive/action_classifier/trusted/action_classifier_trusted_v1.jsonl


In [31]:
# =========================================================
# CELL 22 — TRAIN / TEST SPLIT
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

import numpy as np


# =========================================================
# CONFIG
# =========================================================

TEST_SIZE = 0.20

MAX_SPLIT_ATTEMPTS = 100


# =========================================================
# MULTILABEL BINARIZER
# =========================================================

mlb = MultiLabelBinarizer(
    classes=LABELS
)

Y_all = mlb.fit_transform(
    trusted_v1_df[
        "actions"
    ]
)


# =========================================================
# SPLIT WITH COVERAGE CHECK
# =========================================================

indices = np.arange(
    len(
        trusted_v1_df
    )
)


split_found = False


for attempt in range(
    MAX_SPLIT_ATTEMPTS
):

    split_seed = (
        SEED
        + attempt
    )


    train_idx, test_idx = (
        train_test_split(
            indices,
            test_size=TEST_SIZE,
            random_state=split_seed,
            shuffle=True,
        )
    )


    y_train_check = (
        Y_all[
            train_idx
        ]
    )

    y_test_check = (
        Y_all[
            test_idx
        ]
    )


    train_coverage = (
        y_train_check.sum(
            axis=0
        )
    )

    test_coverage = (
        y_test_check.sum(
            axis=0
        )
    )


    if (
        np.all(
            train_coverage > 0
        )
        and
        np.all(
            test_coverage > 0
        )
    ):

        split_found = True

        break


if not split_found:

    raise RuntimeError(
        "Tidak berhasil membuat split "
        "dengan seluruh label ter-cover."
    )


# =========================================================
# DATAFRAMES
# =========================================================

train_df = (
    trusted_v1_df
    .iloc[
        train_idx
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


test_df = (
    trusted_v1_df
    .iloc[
        test_idx
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# =========================================================
# FINAL Y
# =========================================================

y_train = mlb.transform(
    train_df[
        "actions"
    ]
)

y_test = mlb.transform(
    test_df[
        "actions"
    ]
)


# =========================================================
# DISPLAY COVERAGE
# =========================================================

coverage_rows = []


for idx, label in enumerate(
    LABELS
):

    coverage_rows.append(
        {
            "label":
                label,

            "train":
                int(
                    y_train[
                        :,
                        idx
                    ].sum()
                ),

            "test":
                int(
                    y_test[
                        :,
                        idx
                    ].sum()
                ),
        }
    )


coverage_df = pd.DataFrame(
    coverage_rows
)


print()
print("=" * 80)
print("ACTION CLASSIFIER — TRAIN TEST SPLIT")
print("=" * 80)

print(
    "Split seed :",
    split_seed
)

print(
    "Train      :",
    len(
        train_df
    )
)

print(
    "Test       :",
    len(
        test_df
    )
)

print()

display(
    coverage_df
)

print("=" * 80)


ACTION CLASSIFIER — TRAIN TEST SPLIT
Split seed : 42
Train      : 2533
Test       : 634



,label,train,test
0,READ,1740,420
1,WRITE,665,156
2,DELETE,330,94
3,EXECUTE,779,174
4,NETWORK,371,95
5,INSTALL,62,14
6,PRIVILEGED,264,79
7,SYSTEM_CHANGE,658,192


In [32]:
# =========================================================
# CELL 23 — TRAIN BASELINE
# TF-IDF WORD + CHAR + OVR LOGISTIC REGRESSION
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

import time
import joblib

from sklearn.pipeline import Pipeline
from sklearn.pipeline import FeatureUnion

from sklearn.feature_extraction.text import (
    TfidfVectorizer
)

from sklearn.multiclass import (
    OneVsRestClassifier
)

from sklearn.linear_model import (
    LogisticRegression
)


# =========================================================
# TRAINING TEXT
# =========================================================

X_train = (
    train_df[
        "command"
    ]
    .astype(str)
    .tolist()
)


X_test = (
    test_df[
        "command"
    ]
    .astype(str)
    .tolist()
)


# =========================================================
# WORD TF-IDF
# =========================================================

word_vectorizer = TfidfVectorizer(

    analyzer="word",

    ngram_range=(
        1,
        3
    ),

    min_df=2,

    max_df=0.995,

    sublinear_tf=True,

    max_features=50000,
)


# =========================================================
# CHAR TF-IDF
# =========================================================

char_vectorizer = TfidfVectorizer(

    analyzer="char_wb",

    ngram_range=(
        3,
        6
    ),

    min_df=2,

    sublinear_tf=True,

    max_features=70000,
)


# =========================================================
# FEATURE UNION
# =========================================================

features = FeatureUnion(
    [
        (
            "word",
            word_vectorizer
        ),

        (
            "char",
            char_vectorizer
        ),
    ]
)


# =========================================================
# CLASSIFIER
# =========================================================

classifier = OneVsRestClassifier(

    LogisticRegression(

        C=4.0,

        max_iter=2000,

        class_weight="balanced",

        solver="liblinear",

        random_state=SEED,
    ),

    n_jobs=-1
)


# =========================================================
# PIPELINE
# =========================================================

model = Pipeline(
    [
        (
            "features",
            features
        ),

        (
            "classifier",
            classifier
        ),
    ]
)


# =========================================================
# TRAIN
# =========================================================

print()
print("=" * 80)
print("ACTION CLASSIFIER — BASELINE TRAINING")
print("=" * 80)


start = time.time()


model.fit(
    X_train,
    y_train
)


training_time = (
    time.time()
    - start
)


print(
    "Training complete."
)

print(
    "Training time:",
    round(
        training_time,
        2
    ),
    "sec"
)


# =========================================================
# SAVE JOBLIB
# =========================================================

joblib.dump(
    model,
    MODEL_PATH
)

joblib.dump(
    mlb,
    MLB_PATH
)


print()
print(
    "Model:",
    MODEL_PATH
)

print(
    "MLB  :",
    MLB_PATH
)

print("=" * 80)


ACTION CLASSIFIER — BASELINE TRAINING
Training complete.
Training time: 10.19 sec

Model: /content/drive/MyDrive/action_classifier/models/action_classifier.joblib
MLB  : /content/drive/MyDrive/action_classifier/models/multilabel_binarizer.joblib


In [33]:
# =========================================================
# CELL 24 — INITIAL MODEL EVALUATION
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    hamming_loss,
    classification_report,
    multilabel_confusion_matrix,
)


# =========================================================
# PREDICT
# =========================================================

start = time.time()


y_pred = model.predict(
    X_test
)


prediction_time = (
    time.time()
    - start
)


# =========================================================
# GLOBAL METRICS
# =========================================================

subset_accuracy = accuracy_score(
    y_test,
    y_pred
)


micro_precision = precision_score(
    y_test,
    y_pred,
    average="micro",
    zero_division=0
)


micro_recall = recall_score(
    y_test,
    y_pred,
    average="micro",
    zero_division=0
)


micro_f1 = f1_score(
    y_test,
    y_pred,
    average="micro",
    zero_division=0
)


macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)


weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)


hamming = hamming_loss(
    y_test,
    y_pred
)


# =========================================================
# PRINT METRICS
# =========================================================

print()
print("=" * 80)
print("ACTION CLASSIFIER — INITIAL EVALUATION")
print("=" * 80)

print(
    "Test samples      :",
    len(
        X_test
    )
)

print(
    "Subset accuracy   :",
    round(
        subset_accuracy,
        4
    )
)

print(
    "Micro precision   :",
    round(
        micro_precision,
        4
    )
)

print(
    "Micro recall      :",
    round(
        micro_recall,
        4
    )
)

print(
    "Micro F1          :",
    round(
        micro_f1,
        4
    )
)

print(
    "Macro F1          :",
    round(
        macro_f1,
        4
    )
)

print(
    "Weighted F1       :",
    round(
        weighted_f1,
        4
    )
)

print(
    "Hamming loss      :",
    round(
        hamming,
        4
    )
)

print(
    "Prediction time   :",
    round(
        prediction_time,
        3
    ),
    "sec"
)

print("=" * 80)


# =========================================================
# PER-LABEL REPORT
# =========================================================

print()
print(
    classification_report(
        y_test,
        y_pred,
        target_names=LABELS,
        zero_division=0
    )
)


# =========================================================
# CONFUSION MATRIX PER LABEL
# =========================================================

mcm = multilabel_confusion_matrix(
    y_test,
    y_pred
)


confusion_rows = []


for index, label in enumerate(
    LABELS
):

    tn, fp, fn, tp = (
        mcm[
            index
        ].ravel()
    )


    confusion_rows.append(
        {
            "label":
                label,

            "TP":
                int(tp),

            "FP":
                int(fp),

            "FN":
                int(fn),

            "TN":
                int(tn),
        }
    )


model_confusion_df = (
    pd.DataFrame(
        confusion_rows
    )
)


print()
print("=" * 80)
print("PER-LABEL CONFUSION MATRIX")
print("=" * 80)

display(
    model_confusion_df
)


# =========================================================
# SAVE METRICS
# =========================================================

metrics = {

    "test_samples":
        len(
            X_test
        ),

    "subset_accuracy":
        float(
            subset_accuracy
        ),

    "micro_precision":
        float(
            micro_precision
        ),

    "micro_recall":
        float(
            micro_recall
        ),

    "micro_f1":
        float(
            micro_f1
        ),

    "macro_f1":
        float(
            macro_f1
        ),

    "weighted_f1":
        float(
            weighted_f1
        ),

    "hamming_loss":
        float(
            hamming
        ),

    "training_time_sec":
        float(
            training_time
        ),

    "prediction_time_sec":
        float(
            prediction_time
        ),

    "split_seed":
        int(
            split_seed
        ),
}


with open(
    METRICS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metrics,
        f,
        indent=2
    )


CONFUSION_PATH = (
    f"{REPORT_DIR}/model_confusion_v1.csv"
)


model_confusion_df.to_csv(
    CONFUSION_PATH,
    index=False
)


print()
print(
    "Metrics saved:",
    METRICS_PATH
)

print(
    "Confusion saved:",
    CONFUSION_PATH
)


ACTION CLASSIFIER — INITIAL EVALUATION
Test samples      : 634
Subset accuracy   : 0.5773
Micro precision   : 0.8395
Micro recall      : 0.8333
Micro F1          : 0.8364
Macro F1          : 0.834
Weighted F1       : 0.8393
Hamming loss      : 0.0787
Prediction time   : 0.196 sec

               precision    recall  f1-score   support

         READ       0.92      0.87      0.90       420
        WRITE       0.76      0.79      0.78       156
       DELETE       0.96      0.91      0.93        94
      EXECUTE       0.63      0.71      0.67       174
      NETWORK       0.79      0.80      0.80        95
      INSTALL       0.68      0.93      0.79        14
   PRIVILEGED       0.99      0.95      0.97        79
SYSTEM_CHANGE       0.88      0.81      0.85       192

    micro avg       0.84      0.83      0.84      1224
    macro avg       0.83      0.85      0.83      1224
 weighted avg       0.85      0.83      0.84      1224
  samples avg       0.82      0.84      0.81      1224


,label,TP,FP,FN,TN
0,READ,366,30,54,184
1,WRITE,124,40,32,438
2,DELETE,86,4,8,536
3,EXECUTE,124,73,50,387
4,NETWORK,76,20,19,519
5,INSTALL,13,6,1,614
6,PRIVILEGED,75,1,4,554
7,SYSTEM_CHANGE,156,21,36,421



Metrics saved: /content/drive/MyDrive/action_classifier/reports/training_metrics.json
Confusion saved: /content/drive/MyDrive/action_classifier/reports/model_confusion_v1.csv


In [35]:
# =========================================================
# CELL 25 — ERROR ANALYSIS
# FALSE POSITIVE + FALSE NEGATIVE
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

import pandas as pd
import json


# =========================================================
# BUILD PREDICTION DATAFRAME
# =========================================================

error_rows = []


for i in range(
    len(test_df)
):

    true_labels = set(
        mlb.inverse_transform(
            y_test[i:i+1]
        )[0]
    )

    pred_labels = set(
        mlb.inverse_transform(
            y_pred[i:i+1]
        )[0]
    )


    false_positive = (
        pred_labels
        - true_labels
    )

    false_negative = (
        true_labels
        - pred_labels
    )


    error_rows.append(
        {
            "command":
                X_test[i],

            "true_labels":
                sorted(
                    true_labels
                ),

            "pred_labels":
                sorted(
                    pred_labels
                ),

            "false_positive":
                sorted(
                    false_positive
                ),

            "false_negative":
                sorted(
                    false_negative
                ),

            "exact_match":
                true_labels
                ==
                pred_labels,
        }
    )


error_analysis_df = pd.DataFrame(
    error_rows
)


# =========================================================
# SUMMARY
# =========================================================

print()
print("=" * 80)
print("ACTION CLASSIFIER — ERROR ANALYSIS")
print("=" * 80)

print(
    "Test samples :",
    len(
        error_analysis_df
    )
)

print(
    "Exact        :",
    int(
        error_analysis_df[
            "exact_match"
        ].sum()
    )
)

print(
    "Errors       :",
    int(
        (
            ~error_analysis_df[
                "exact_match"
            ]
        ).sum()
    )
)


# =========================================================
# PER-LABEL FP / FN
# =========================================================

for label in LABELS:

    fp_count = (
        error_analysis_df[
            "false_positive"
        ]
        .apply(
            lambda x:
                label in x
        )
        .sum()
    )

    fn_count = (
        error_analysis_df[
            "false_negative"
        ]
        .apply(
            lambda x:
                label in x
        )
        .sum()
    )


    print(
        f"{label:14}",
        f"FP={fp_count:4}",
        f"FN={fn_count:4}"
    )


# =========================================================
# SAVE
# =========================================================

save_error_df = (
    error_analysis_df.copy()
)


for column in [
    "true_labels",
    "pred_labels",
    "false_positive",
    "false_negative",
]:

    save_error_df[
        column
    ] = save_error_df[
        column
    ].apply(
        json.dumps
    )


save_error_df.to_csv(
    ERROR_ANALYSIS_PATH,
    index=False
)


print()
print(
    "Saved:",
    ERROR_ANALYSIS_PATH
)

print("=" * 80)


ACTION CLASSIFIER — ERROR ANALYSIS
Test samples : 634
Exact        : 366
Errors       : 268
READ           FP=  30 FN=  54
WRITE          FP=  40 FN=  32
DELETE         FP=   4 FN=   8
EXECUTE        FP=  73 FN=  50
NETWORK        FP=  20 FN=  19
INSTALL        FP=   6 FN=   1
PRIVILEGED     FP=   1 FN=   4
SYSTEM_CHANGE  FP=  21 FN=  36

Saved: /content/drive/MyDrive/action_classifier/reports/error_analysis.csv


In [36]:
# =========================================================
# CELL 26 — EXECUTE ERROR INSPECTION
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

TARGET_LABEL = "EXECUTE"


execute_fp_df = (
    error_analysis_df[
        error_analysis_df[
            "false_positive"
        ].apply(
            lambda x:
                TARGET_LABEL in x
        )
    ]
    .copy()
)


execute_fn_df = (
    error_analysis_df[
        error_analysis_df[
            "false_negative"
        ].apply(
            lambda x:
                TARGET_LABEL in x
        )
    ]
    .copy()
)


print()
print("=" * 90)
print("EXECUTE — FALSE POSITIVES")
print("=" * 90)

print(
    "Count:",
    len(
        execute_fp_df
    )
)


display(
    execute_fp_df[
        [
            "command",
            "true_labels",
            "pred_labels",
        ]
    ].head(40)
)


print()
print("=" * 90)
print("EXECUTE — FALSE NEGATIVES")
print("=" * 90)

print(
    "Count:",
    len(
        execute_fn_df
    )
)


display(
    execute_fn_df[
        [
            "command",
            "true_labels",
            "pred_labels",
        ]
    ].head(40)
)


EXECUTE — FALSE POSITIVES
Count: 73


,command,true_labels,pred_labels
0,"printf ""username_1:new_password_1\nusername_2:...","[PRIVILEGED, SYSTEM_CHANGE, WRITE]","[EXECUTE, PRIVILEGED]"
13,for file in *.py; do sed -i '1i #!/usr/bin/pyt...,"[SYSTEM_CHANGE, WRITE]","[EXECUTE, READ, WRITE]"
23,catimg -r 2 path/to/file,[READ],"[EXECUTE, READ, WRITE]"
28,sudo useradd -s|--shell path/to/shell username,"[PRIVILEGED, SYSTEM_CHANGE]","[EXECUTE, PRIVILEGED, SYSTEM_CHANGE]"
38,find . -exec rm '{}' \;,"[DELETE, READ]","[DELETE, EXECUTE, READ]"
49,aws s3api put-bucket-policy --bucket bucket_na...,"[NETWORK, SYSTEM_CHANGE, WRITE]","[EXECUTE, READ]"
52,lsd --tree -d,[READ],[EXECUTE]
53,rekor-cli search --artifact path/to/file.ext,[READ],"[EXECUTE, READ]"
88,shuf -i 1-100 -n 1 | xargs curl -O http://www....,"[NETWORK, READ, WRITE]","[EXECUTE, NETWORK, WRITE]"
92,sudo tlmgr install package,"[INSTALL, PRIVILEGED, SYSTEM_CHANGE]","[EXECUTE, INSTALL, NETWORK, PRIVILEGED, SYSTEM..."



EXECUTE — FALSE NEGATIVES
Count: 50


,command,true_labels,pred_labels
19,swipl,[EXECUTE],[]
25,cbatticon --list-icon-types,[EXECUTE],[READ]
35,find ~ -iregex '.*\(.sh\|.bash\)$' | xargs gre...,"[EXECUTE, READ]",[READ]
42,find / -type d | shuf -n 1 | xargs rm -rf,"[DELETE, EXECUTE, READ]","[DELETE, READ]"
45,sudo rtcwake -m freeze --date YYYYMMDDhhmm,"[EXECUTE, PRIVILEGED, SYSTEM_CHANGE]",[PRIVILEGED]
47,starship init bash|elvish|fish|ion|powershell|...,[EXECUTE],[WRITE]
50,apt-clone restore path/to/backup.tar.gz,"[EXECUTE, SYSTEM_CHANGE]","[READ, SYSTEM_CHANGE, WRITE]"
54,`curl http://example.com/ | sed -e 's/[^a-zA-Z...,"[EXECUTE, NETWORK, READ]","[NETWORK, READ]"
61,curl -L 'http://www.example.com' | shuf -n 1,"[EXECUTE, NETWORK]",[NETWORK]
78,find `pwd` -type d -exec rm -rvf {} \;,"[DELETE, EXECUTE, READ]","[DELETE, READ]"


In [39]:
# =========================================================
# CELL 27 — PREDICTION PROBABILITY ANALYSIS
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

import numpy as np
import pandas as pd


# =========================================================
# GET PROBABILITIES
# =========================================================

y_prob = model.predict_proba(
    X_test
)


# =========================================================
# PER-LABEL PROBABILITY SUMMARY
# =========================================================

probability_rows = []


for index, label in enumerate(
    LABELS
):

    positive_probs = (
        y_prob[
            y_test[:, index] == 1,
            index
        ]
    )

    negative_probs = (
        y_prob[
            y_test[:, index] == 0,
            index
        ]
    )


    probability_rows.append(
        {
            "label":
                label,

            "positive_mean":
                float(
                    np.mean(
                        positive_probs
                    )
                )
                if len(
                    positive_probs
                )
                else 0,

            "negative_mean":
                float(
                    np.mean(
                        negative_probs
                    )
                )
                if len(
                    negative_probs
                )
                else 0,

            "positive_median":
                float(
                    np.median(
                        positive_probs
                    )
                )
                if len(
                    positive_probs
                )
                else 0,

            "negative_median":
                float(
                    np.median(
                        negative_probs
                    )
                )
                if len(
                    negative_probs
                )
                else 0,
        }
    )


probability_df = pd.DataFrame(
    probability_rows
)


print()
print("=" * 80)
print("ACTION CLASSIFIER — PROBABILITY ANALYSIS")
print("=" * 80)

display(
    probability_df
)

print("=" * 80)


ACTION CLASSIFIER — PROBABILITY ANALYSIS


,label,positive_mean,negative_mean,positive_median,negative_median
0,READ,0.829360,0.230424,0.935381,0.128124
1,WRITE,0.757059,0.155940,0.911422,0.081322
2,DELETE,0.879068,0.039925,0.978873,0.023222
3,EXECUTE,0.662245,0.234222,0.733114,0.149625
4,NETWORK,0.796827,0.086121,0.942896,0.028614
5,INSTALL,0.898084,0.028443,0.982945,0.008107
6,PRIVILEGED,0.931258,0.027825,0.992467,0.016519
7,SYSTEM_CHANGE,0.783995,0.109508,0.935597,0.049346


In [40]:
# =========================================================
# CELL 29 — BUILD WEAKNESS SUMMARY
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

from collections import Counter
import pandas as pd
import json


# =========================================================
# CONFIG
# =========================================================

WEAK_LABELS = [
    "EXECUTE",
    "WRITE",
    "SYSTEM_CHANGE",
    "INSTALL",
]


# =========================================================
# COLLECT ERROR PATTERNS
# =========================================================

weakness_rows = []


for label in WEAK_LABELS:

    fp_df = error_analysis_df[
        error_analysis_df[
            "false_positive"
        ].apply(
            lambda x:
                label in x
        )
    ]

    fn_df = error_analysis_df[
        error_analysis_df[
            "false_negative"
        ].apply(
            lambda x:
                label in x
        )
    ]


    weakness_rows.append(
        {
            "label": label,
            "false_positive": len(fp_df),
            "false_negative": len(fn_df),
            "total_errors":
                len(fp_df)
                + len(fn_df),
        }
    )


weakness_summary_df = (
    pd.DataFrame(
        weakness_rows
    )
    .sort_values(
        "total_errors",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# =========================================================
# SHOW EXAMPLES
# =========================================================

print()
print("=" * 90)
print("ACTION CLASSIFIER — WEAKNESS SUMMARY")
print("=" * 90)

display(
    weakness_summary_df
)


for label in WEAK_LABELS:

    print()
    print("=" * 90)
    print(
        f"{label} — ERROR EXAMPLES"
    )
    print("=" * 90)


    sample_errors = error_analysis_df[
        error_analysis_df.apply(
            lambda row:
                label
                in row[
                    "false_positive"
                ]
                or
                label
                in row[
                    "false_negative"
                ],
            axis=1
        )
    ]


    display(
        sample_errors[
            [
                "command",
                "true_labels",
                "pred_labels",
                "false_positive",
                "false_negative",
            ]
        ].head(15)
    )


# =========================================================
# SAVE
# =========================================================

WEAKNESS_PATH = (
    f"{REPORT_DIR}/weakness_summary_v1.csv"
)


weakness_summary_df.to_csv(
    WEAKNESS_PATH,
    index=False
)


print()
print(
    "Saved:",
    WEAKNESS_PATH
)


ACTION CLASSIFIER — WEAKNESS SUMMARY


,label,false_positive,false_negative,total_errors
0,EXECUTE,73,50,123
1,WRITE,40,32,72
2,SYSTEM_CHANGE,21,36,57
3,INSTALL,6,1,7



EXECUTE — ERROR EXAMPLES


,command,true_labels,pred_labels,false_positive,false_negative
0,"printf ""username_1:new_password_1\nusername_2:...","[PRIVILEGED, SYSTEM_CHANGE, WRITE]","[EXECUTE, PRIVILEGED]",[EXECUTE],"[SYSTEM_CHANGE, WRITE]"
13,for file in *.py; do sed -i '1i #!/usr/bin/pyt...,"[SYSTEM_CHANGE, WRITE]","[EXECUTE, READ, WRITE]","[EXECUTE, READ]",[SYSTEM_CHANGE]
19,swipl,[EXECUTE],[],[],[EXECUTE]
23,catimg -r 2 path/to/file,[READ],"[EXECUTE, READ, WRITE]","[EXECUTE, WRITE]",[]
25,cbatticon --list-icon-types,[EXECUTE],[READ],[READ],[EXECUTE]
28,sudo useradd -s|--shell path/to/shell username,"[PRIVILEGED, SYSTEM_CHANGE]","[EXECUTE, PRIVILEGED, SYSTEM_CHANGE]",[EXECUTE],[]
35,find ~ -iregex '.*\(.sh\|.bash\)$' | xargs gre...,"[EXECUTE, READ]",[READ],[],[EXECUTE]
38,find . -exec rm '{}' \;,"[DELETE, READ]","[DELETE, EXECUTE, READ]",[EXECUTE],[]
42,find / -type d | shuf -n 1 | xargs rm -rf,"[DELETE, EXECUTE, READ]","[DELETE, READ]",[],[EXECUTE]
45,sudo rtcwake -m freeze --date YYYYMMDDhhmm,"[EXECUTE, PRIVILEGED, SYSTEM_CHANGE]",[PRIVILEGED],[],"[EXECUTE, SYSTEM_CHANGE]"



WRITE — ERROR EXAMPLES


...


SYSTEM_CHANGE — ERROR EXAMPLES


,command,true_labels,pred_labels,false_positive,false_negative
0,"printf ""username_1:new_password_1\nusername_2:...","[PRIVILEGED, SYSTEM_CHANGE, WRITE]","[EXECUTE, PRIVILEGED]",[EXECUTE],"[SYSTEM_CHANGE, WRITE]"
13,for file in *.py; do sed -i '1i #!/usr/bin/pyt...,"[SYSTEM_CHANGE, WRITE]","[EXECUTE, READ, WRITE]","[EXECUTE, READ]",[SYSTEM_CHANGE]
17,find -iname '*.*' | xargs sed -i 's/\t/\ /g',"[READ, SYSTEM_CHANGE, WRITE]","[READ, WRITE]",[],[SYSTEM_CHANGE]
20,sudo repquota --human-readable filesystem,"[PRIVILEGED, READ]","[PRIVILEGED, SYSTEM_CHANGE]",[SYSTEM_CHANGE],[READ]
33,grant action_list on object_type object_name t...,[SYSTEM_CHANGE],[],[],[SYSTEM_CHANGE]
40,xmodmap path/to/file,"[READ, SYSTEM_CHANGE]",[WRITE],[WRITE],"[READ, SYSTEM_CHANGE]"
45,sudo rtcwake -m freeze --date YYYYMMDDhhmm,"[EXECUTE, PRIVILEGED, SYSTEM_CHANGE]",[PRIVILEGED],[],"[EXECUTE, SYSTEM_CHANGE]"
49,aws s3api put-bucket-policy --bucket bucket_na...,"[NETWORK, SYSTEM_CHANGE, WRITE]","[EXECUTE, READ]","[EXECUTE, READ]","[NETWORK, SYSTEM_CHANGE, WRITE]"
62,trizen -Syua,"[INSTALL, NETWORK, SYSTEM_CHANGE]",[],[],"[INSTALL, NETWORK, SYSTEM_CHANGE]"
66,sudo machinectl shell machine_name,"[EXECUTE, PRIVILEGED, SYSTEM_CHANGE]","[EXECUTE, PRIVILEGED]",[],[SYSTEM_CHANGE]



INSTALL — ERROR EXAMPLES


,command,true_labels,pred_labels,false_positive,false_negative
34,stack new package template,[WRITE],[INSTALL],[INSTALL],[WRITE]
62,trizen -Syua,"[INSTALL, NETWORK, SYSTEM_CHANGE]",[],[],"[INSTALL, NETWORK, SYSTEM_CHANGE]"
224,tldr pacman remove,[READ],"[INSTALL, READ]",[INSTALL],[]
331,sudo nala fetch,"[PRIVILEGED, SYSTEM_CHANGE]","[INSTALL, NETWORK, PRIVILEGED, SYSTEM_CHANGE]","[INSTALL, NETWORK]",[]
522,npm unstar package_name,"[NETWORK, SYSTEM_CHANGE]","[INSTALL, NETWORK, SYSTEM_CHANGE, WRITE]","[INSTALL, WRITE]",[]
550,npm bugs package_name,"[NETWORK, READ]","[INSTALL, NETWORK, SYSTEM_CHANGE, WRITE]","[INSTALL, SYSTEM_CHANGE, WRITE]",[READ]
605,sudo pacman -Rsn package,"[DELETE, PRIVILEGED, SYSTEM_CHANGE]","[DELETE, INSTALL, PRIVILEGED, SYSTEM_CHANGE]",[INSTALL],[]



Saved: /content/drive/MyDrive/action_classifier/reports/weakness_summary_v1.csv


In [41]:
# =========================================================
# CELL 30 — TEACHER WEAKNESS AUDIT
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

import json
import requests
import time


# =========================================================
# BUILD ERROR SAMPLE SET
# =========================================================

AUDIT_ERROR_PER_LABEL = 12

teacher_error_examples = []


for label in WEAK_LABELS:

    candidates = error_analysis_df[
        error_analysis_df.apply(
            lambda row:
                label
                in row[
                    "false_positive"
                ]
                or
                label
                in row[
                    "false_negative"
                ],
            axis=1
        )
    ]


    if len(candidates) == 0:
        continue


    sample_n = min(
        AUDIT_ERROR_PER_LABEL,
        len(candidates)
    )


    sampled = candidates.sample(
        n=sample_n,
        random_state=SEED
    )


    for _, row in sampled.iterrows():

        teacher_error_examples.append(
            {
                "focus_label": label,
                "command":
                    row["command"],
                "ground_truth":
                    row["true_labels"],
                "model_prediction":
                    row["pred_labels"],
                "false_positive":
                    row["false_positive"],
                "false_negative":
                    row["false_negative"],
            }
        )


# =========================================================
# AUDIT PROMPT
# =========================================================

REPAIR_AUDIT_SYSTEM_PROMPT = """
You are auditing a multi-label shell-command classifier.

You are NOT executing commands.

The classifier labels are:

READ
WRITE
DELETE
EXECUTE
NETWORK
INSTALL
PRIVILEGED
SYSTEM_CHANGE

Your job is to identify recurring classification
boundaries that explain the model errors.

IMPORTANT:

EXECUTE does NOT mean that every shell utility is executed.

Use EXECUTE only when the command's semantic action
substantially involves launching or invoking executable
code, scripts, programs, interpreters, services,
or another command as an important action.

Do not automatically assign EXECUTE merely because:
- a shell command itself runs
- find uses -exec
- a normal utility processes data
- a command performs READ or WRITE

SYSTEM_CHANGE means persistent or meaningful system-state
or configuration modification.

WRITE means creation, copying, movement, overwrite,
append, or local persistence of data.

INSTALL means software/package/dependency installation,
upgrade, or removal.

Analyze error patterns.

Return ONLY JSON.

Schema:

{
  "weaknesses": [
    {
      "label": "LABEL",
      "problem": "short description",
      "boundary_rule": "short classification rule"
    }
  ]
}

Do not return training examples yet.
"""


audit_payload = json.dumps(
    teacher_error_examples,
    ensure_ascii=False
)


headers = {
    "Authorization":
        f"Bearer {API_KEY}",

    "Content-Type":
        "application/json",
}


payload = {
    "model": MODEL,

    "temperature": 0,

    "thinking": {
        "type": "disabled"
    },

    "response_format": {
        "type": "json_object"
    },

    "messages": [
        {
            "role": "system",
            "content":
                REPAIR_AUDIT_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content":
                audit_payload,
        },
    ],
}


response = requests.post(
    BASE_URL,
    headers=headers,
    json=payload,
    timeout=(
        TIMEOUT_CONNECT,
        TIMEOUT_READ
    ),
)

response.raise_for_status()


repair_audit = (
    response
    .json()[
        "choices"
    ][0][
        "message"
    ][
        "content"
    ]
)


repair_audit_data = json.loads(
    repair_audit
)


print()
print("=" * 90)
print("TEACHER — WEAKNESS AUDIT")
print("=" * 90)

print(
    json.dumps(
        repair_audit_data,
        indent=2,
        ensure_ascii=False
    )
)

print("=" * 90)


# =========================================================
# SAVE
# =========================================================

REPAIR_AUDIT_PATH = (
    f"{REPORT_DIR}/teacher_repair_audit_v1.json"
)


with open(
    REPAIR_AUDIT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        repair_audit_data,
        f,
        indent=2,
        ensure_ascii=False
    )


print(
    "Saved:",
    REPAIR_AUDIT_PATH
)


TEACHER — WEAKNESS AUDIT
{
  "weaknesses": [
    {
      "label": "EXECUTE",
      "problem": "Model over-applies EXECUTE to commands that merely run a utility or process data, and under-applies it to commands that launch daemons or scripts.",
      "boundary_rule": "EXECUTE should be reserved for commands whose primary purpose is to launch or run executable code, scripts, interpreters, or services, not for general data processing or utility invocation."
    },
    {
      "label": "WRITE",
      "problem": "Model often misses WRITE when a command modifies system state or configuration, and sometimes labels read-only operations as WRITE.",
      "boundary_rule": "WRITE should be assigned when the command creates, modifies, or persists data locally, including configuration changes, but not for mere reads or network operations."
    },
    {
      "label": "SYSTEM_CHANGE",
      "problem": "Model frequently fails to recognize persistent system-state modifications, especially when the co

In [42]:
# =========================================================
# CELL 31 — TEACHER TARGETED HARD-CASE GENERATION
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

import json
import time
import requests


# =========================================================
# GENERATION CONFIG
# =========================================================

REPAIR_TARGETS = {

    "EXECUTE": 200,

    "WRITE": 100,

    "SYSTEM_CHANGE": 100,

    "INSTALL": 100,
}


GENERATION_BATCH_SIZE = 25

GENERATION_DELAY = 0.25


# =========================================================
# GENERATION PROMPT
# =========================================================

REPAIR_GENERATION_SYSTEM_PROMPT = """
You generate synthetic training examples for a
multi-label shell-command action classifier.

You are NOT executing commands.

Allowed labels:

READ
WRITE
DELETE
EXECUTE
NETWORK
INSTALL
PRIVILEGED
SYSTEM_CHANGE

Generate NEW shell-command examples that clarify
classification boundaries.

Do NOT copy commands supplied in previous audit data.

Use safe placeholders where appropriate:

<TEST_URL>
<LAB_HOST>
<TEMP_DIR>
<TEST_FILE>
<USER_NAME>
<PACKAGE_NAME>

Important taxonomy:

READ:
inspect, query, display, search, or retrieve local data.

WRITE:
create, copy, move, overwrite, append, download,
or otherwise persist local data.

DELETE:
remove files/directories/data.

EXECUTE:
launch or invoke executable code, scripts,
interpreters, programs, services, or another command
as an important semantic action.

Do NOT assign EXECUTE merely because:
- every shell command technically runs
- find contains -exec
- a normal utility transforms files
- a command only reads or writes data

NETWORK:
connect, query, transmit, upload, download,
or otherwise interact with remote/network systems.

INSTALL:
install, upgrade, remove, or manage software packages
or dependencies.

PRIVILEGED:
explicit elevated privilege boundary such as sudo.

SYSTEM_CHANGE:
modify system configuration, permissions, services,
users, packages, mounts, firewall state,
or other operating-system state.

Generate boundary examples:
- positive examples for the target label
- negative near-miss examples that look similar
- multi-label examples
- chained command examples

Return ONLY JSON.

Schema:

{
  "examples": [
    {
      "command": "...",
      "description": "...",
      "actions": ["LABEL"],
      "confidence": 0.95,
      "ambiguous": false
    }
  ]
}

All generated examples must be unambiguous.

Do not include explanations.
"""


# =========================================================
# CALL GENERATOR
# =========================================================

def generate_repair_batch(
    focus_label,
    batch_size
):

    weakness_info = [
        item
        for item in repair_audit_data.get(
            "weaknesses",
            []
        )
        if item.get(
            "label"
        ) == focus_label
    ]


    user_prompt = {
        "focus_label":
            focus_label,

        "count":
            batch_size,

        "known_weakness":
            weakness_info,

        "requirements": [
            "Generate new commands",
            "Do not copy audit commands",
            "Include both positive and near-miss examples",
            "Use multiple shell utilities",
            "Include some multi-label examples",
        ],
    }


    payload = {

        "model":
            MODEL,

        "temperature":
            0.4,

        "thinking": {
            "type":
                "disabled"
        },

        "response_format": {
            "type":
                "json_object"
        },

        "messages": [

            {
                "role":
                    "system",

                "content":
                    REPAIR_GENERATION_SYSTEM_PROMPT,
            },

            {
                "role":
                    "user",

                "content":
                    json.dumps(
                        user_prompt,
                        ensure_ascii=False
                    ),
            },
        ],
    }


    response = requests.post(

        BASE_URL,

        headers={
            "Authorization":
                f"Bearer {API_KEY}",

            "Content-Type":
                "application/json",
        },

        json=payload,

        timeout=(
            TIMEOUT_CONNECT,
            TIMEOUT_READ
        ),
    )


    response.raise_for_status()


    content = (
        response
        .json()[
            "choices"
        ][0][
            "message"
        ][
            "content"
        ]
    )


    return json.loads(
        content
    )


# =========================================================
# GENERATION LOOP
# =========================================================

generated_repair_examples = []


for focus_label, target_count in (
    REPAIR_TARGETS.items()
):

    print()
    print("=" * 80)

    print(
        "Generating:",
        focus_label,
        "target:",
        target_count
    )

    print("=" * 80)


    collected = 0


    while collected < target_count:

        batch_size = min(
            GENERATION_BATCH_SIZE,
            target_count
            - collected
        )


        try:

            result = (
                generate_repair_batch(
                    focus_label,
                    batch_size
                )
            )


            examples = result.get(
                "examples",
                []
            )


            for example in examples:

                example[
                    "focus_label"
                ] = focus_label

                example[
                    "source"
                ] = (
                    "teacher_targeted_repair"
                )

                generated_repair_examples.append(
                    example
                )


            collected += len(
                examples
            )


            print(
                f"{focus_label}: "
                f"{collected}/{target_count}"
            )


            if len(examples) == 0:

                print(
                    "WARNING: empty generation batch"
                )

                break


        except Exception as e:

            print(
                "Generation error:",
                str(e)[:200]
            )


        time.sleep(
            GENERATION_DELAY
        )


print()
print("=" * 80)

print(
    "Generated total:",
    len(
        generated_repair_examples
    )
)

print("=" * 80)


Generating: EXECUTE target: 200
EXECUTE: 25/200
EXECUTE: 50/200
EXECUTE: 75/200
EXECUTE: 100/200
EXECUTE: 125/200
EXECUTE: 150/200
EXECUTE: 175/200
EXECUTE: 200/200

Generating: WRITE target: 100
WRITE: 25/100
WRITE: 50/100
WRITE: 75/100
WRITE: 100/100

Generating: SYSTEM_CHANGE target: 100
SYSTEM_CHANGE: 25/100
SYSTEM_CHANGE: 50/100
SYSTEM_CHANGE: 75/100
SYSTEM_CHANGE: 100/100

Generating: INSTALL target: 100
INSTALL: 25/100
INSTALL: 50/100
Generation error: Unterminated string starting at: line 1076 column 22 (char 30778)
INSTALL: 75/100
INSTALL: 100/100

Generated total: 500


In [47]:
# =========================================================
# CELL 32 — BUILD TARGETED REPAIR DATASET
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

import pandas as pd
import json


# =========================================================
# CREATE DATAFRAME
# =========================================================

repair_df = pd.DataFrame(
    generated_repair_examples
)


print()
print("=" * 80)
print("ACTION CLASSIFIER — REPAIR DATASET BUILD")
print("=" * 80)

print(
    "Generated raw:",
    len(
        repair_df
    )
)


# =========================================================
# BASIC REQUIRED FIELDS
# =========================================================

required_fields = [
    "command",
    "actions",
    "confidence",
    "ambiguous",
]


for field in required_fields:

    if field not in repair_df.columns:

        raise RuntimeError(
            f"Missing repair field: {field}"
        )


# =========================================================
# CLEAN COMMAND
# =========================================================

repair_df[
    "command"
] = (
    repair_df[
        "command"
    ]
    .astype(str)
    .str.strip()
)


repair_df = repair_df[
    repair_df[
        "command"
    ].str.len() > 0
].copy()


# =========================================================
# VALIDATE LABELS
# =========================================================

repair_df[
    "_valid_actions"
] = repair_df[
    "actions"
].apply(
    validate_actions
)


# =========================================================
# TRUST FILTER
# =========================================================

repair_df = repair_df[

    repair_df[
        "_valid_actions"
    ]

    &

    (
        repair_df[
            "confidence"
        ] >= 0.90
    )

    &

    (
        repair_df[
            "ambiguous"
        ] == False
    )

].copy()


# =========================================================
# INTERNAL DEDUP
# =========================================================

before_internal_dedup = len(
    repair_df
)


repair_df[
    "_command_key"
] = (
    repair_df[
        "command"
    ]
    .str.lower()
)


repair_df = (
    repair_df
    .drop_duplicates(
        subset=[
            "_command_key"
        ]
    )
    .copy()
)


internal_duplicates = (
    before_internal_dedup
    - len(
        repair_df
    )
)


# =========================================================
# REMOVE OVERLAP WITH EXISTING TRAIN DATA
# =========================================================

existing_commands = set(

    trusted_v1_df[
        "command"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)


overlap_mask = (
    repair_df[
        "_command_key"
    ].isin(
        existing_commands
    )
)


overlap_count = int(
    overlap_mask.sum()
)


repair_df = (
    repair_df[
        ~overlap_mask
    ]
    .copy()
)


# =========================================================
# FINAL CLEANUP
# =========================================================

repair_df = (
    repair_df
    .drop(
        columns=[
            "_valid_actions",
            "_command_key",
        ]
    )
    .reset_index(
        drop=True
    )
)


repair_df[
    "split_origin"
] = (
    "teacher_targeted_repair"
)


# =========================================================
# SAVE
# =========================================================

repair_df.to_json(
    REPAIR_DATASET_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)


# =========================================================
# LABEL DISTRIBUTION
# =========================================================

repair_label_counts = {
    label: 0
    for label in LABELS
}


for actions in repair_df[
    "actions"
]:

    for label in actions:

        repair_label_counts[
            label
        ] += 1


repair_distribution_df = (
    pd.DataFrame(
        [
            {
                "label":
                    label,

                "count":
                    count,
            }

            for label, count
            in repair_label_counts.items()
        ]
    )
)


# =========================================================
# STATUS
# =========================================================

print()
print("=" * 80)
print("TARGETED REPAIR DATASET")
print("=" * 80)

print(
    "Raw generated       :",
    len(
        generated_repair_examples
    )
)

print(
    "Internal duplicates :",
    internal_duplicates
)

print(
    "Existing overlap    :",
    overlap_count
)

print(
    "Final repair        :",
    len(
        repair_df
    )
)

print()

display(
    repair_distribution_df
)

print()

print(
    "Saved:",
    REPAIR_DATASET_PATH
)

print("=" * 80)


ACTION CLASSIFIER — REPAIR DATASET BUILD
Generated raw: 500

TARGETED REPAIR DATASET
Raw generated       : 500
Internal duplicates : 173
Existing overlap    : 6
Final repair        : 319



,label,count
0,READ,58
1,WRITE,83
2,DELETE,16
3,EXECUTE,39
4,NETWORK,51
5,INSTALL,50
6,PRIVILEGED,78
7,SYSTEM_CHANGE,121



Saved: /content/drive/MyDrive/action_classifier/repair/targeted_repair_v1.jsonl


In [48]:
# =========================================================
# CELL 33 — MERGE TRUSTED V1 + TARGETED REPAIR
# MODEL 01 — ACTION CLASSIFIER V2
# =========================================================

import pandas as pd


# =========================================================
# PREPARE BASE DATASET
# =========================================================

base_v2_df = (
    trusted_v1_df.copy()
)


# =========================================================
# PREPARE REPAIR DATASET
# =========================================================

repair_v2_df = (
    repair_df.copy()
)


# =========================================================
# ALIGN COLUMNS
# =========================================================

all_columns = set(
    base_v2_df.columns
).union(
    repair_v2_df.columns
)


for column in all_columns:

    if column not in base_v2_df.columns:
        base_v2_df[column] = None

    if column not in repair_v2_df.columns:
        repair_v2_df[column] = None


# =========================================================
# MERGE
# =========================================================

trusted_v2_df = pd.concat(
    [
        base_v2_df,
        repair_v2_df,
    ],
    ignore_index=True
)


# =========================================================
# FINAL DEDUP
# =========================================================

trusted_v2_df[
    "_command_key"
] = (
    trusted_v2_df[
        "command"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)


before_dedup = len(
    trusted_v2_df
)


trusted_v2_df = (
    trusted_v2_df
    .drop_duplicates(
        subset=[
            "_command_key"
        ],
        keep="first"
    )
    .drop(
        columns=[
            "_command_key"
        ]
    )
    .reset_index(
        drop=True
    )
)


after_dedup = len(
    trusted_v2_df
)


# =========================================================
# VALIDATE
# =========================================================

invalid_count = int(
    (
        ~trusted_v2_df[
            "actions"
        ].apply(
            validate_actions
        )
    ).sum()
)


assert (
    invalid_count == 0
), f"Invalid labels: {invalid_count}"


# =========================================================
# SAVE
# =========================================================

TRUSTED_V2_PATH = (
    f"{TRUSTED_DIR}/action_classifier_trusted_v2.jsonl"
)


trusted_v2_df.to_json(
    TRUSTED_V2_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)


# =========================================================
# STATUS
# =========================================================

print()
print("=" * 80)
print("ACTION CLASSIFIER — TRUSTED V2")
print("=" * 80)

print(
    "Trusted V1       :",
    len(
        trusted_v1_df
    )
)

print(
    "Repair           :",
    len(
        repair_v2_df
    )
)

print(
    "Before dedup     :",
    before_dedup
)

print(
    "After dedup      :",
    after_dedup
)

print(
    "Added net samples:",
    after_dedup
    - len(
        trusted_v1_df
    )
)

print(
    "Saved            :",
    TRUSTED_V2_PATH
)

print("=" * 80)


ACTION CLASSIFIER — TRUSTED V2
Trusted V1       : 3167
Repair           : 319
Before dedup     : 3486
After dedup      : 3486
Added net samples: 319
Saved            : /content/drive/MyDrive/action_classifier/trusted/action_classifier_trusted_v2.jsonl


In [49]:
# =========================================================
# CELL 34 — TRAIN / VALIDATION / BLIND SPLIT
# MODEL 01 — ACTION CLASSIFIER V2
# =========================================================

import numpy as np

from sklearn.model_selection import (
    train_test_split
)

from sklearn.preprocessing import (
    MultiLabelBinarizer
)


# =========================================================
# CONFIG
# =========================================================

TRAIN_RATIO = 0.70
VALID_RATIO = 0.15
BLIND_RATIO = 0.15


# =========================================================
# MULTI-LABEL ENCODER
# =========================================================

mlb_v2 = MultiLabelBinarizer(
    classes=LABELS
)


Y_v2 = mlb_v2.fit_transform(
    trusted_v2_df[
        "actions"
    ]
)


indices = np.arange(
    len(
        trusted_v2_df
    )
)


# =========================================================
# FIRST SPLIT
# TRAIN 70%
# TEMP 30%
# =========================================================

train_idx_v2, temp_idx_v2 = (
    train_test_split(
        indices,
        test_size=(
            VALID_RATIO
            + BLIND_RATIO
        ),
        random_state=SEED,
        shuffle=True,
    )
)


# =========================================================
# SECOND SPLIT
# VALID 15%
# BLIND 15%
# =========================================================

valid_idx_v2, blind_idx_v2 = (
    train_test_split(
        temp_idx_v2,
        test_size=0.50,
        random_state=SEED + 1,
        shuffle=True,
    )
)


# =========================================================
# DATAFRAMES
# =========================================================

train_v2_df = (
    trusted_v2_df
    .iloc[
        train_idx_v2
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


valid_v2_df = (
    trusted_v2_df
    .iloc[
        valid_idx_v2
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


blind_v2_df = (
    trusted_v2_df
    .iloc[
        blind_idx_v2
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# =========================================================
# LABEL MATRICES
# =========================================================

y_train_v2 = mlb_v2.transform(
    train_v2_df[
        "actions"
    ]
)

y_valid_v2 = mlb_v2.transform(
    valid_v2_df[
        "actions"
    ]
)

y_blind_v2 = mlb_v2.transform(
    blind_v2_df[
        "actions"
    ]
)


# =========================================================
# COVERAGE REPORT
# =========================================================

coverage_rows = []


for i, label in enumerate(
    LABELS
):

    coverage_rows.append(
        {
            "label":
                label,

            "train":
                int(
                    y_train_v2[
                        :,
                        i
                    ].sum()
                ),

            "valid":
                int(
                    y_valid_v2[
                        :,
                        i
                    ].sum()
                ),

            "blind":
                int(
                    y_blind_v2[
                        :,
                        i
                    ].sum()
                ),
        }
    )


coverage_v2_df = pd.DataFrame(
    coverage_rows
)


print()
print("=" * 80)
print("ACTION CLASSIFIER V2 — DATA SPLIT")
print("=" * 80)

print(
    "Total :",
    len(
        trusted_v2_df
    )
)

print(
    "Train :",
    len(
        train_v2_df
    )
)

print(
    "Valid :",
    len(
        valid_v2_df
    )
)

print(
    "Blind :",
    len(
        blind_v2_df
    )
)

print()

display(
    coverage_v2_df
)

print("=" * 80)


ACTION CLASSIFIER V2 — DATA SPLIT
Total : 3486
Train : 2440
Valid : 523
Blind : 523



,label,train,valid,blind
0,READ,1590,324,304
1,WRITE,646,135,123
2,DELETE,310,61,69
3,EXECUTE,698,141,153
4,NETWORK,357,75,85
5,INSTALL,80,20,26
6,PRIVILEGED,290,58,73
7,SYSTEM_CHANGE,640,170,161


In [50]:
# =========================================================
# CELL 35 — TRAIN ACTION CLASSIFIER V2
# SAME ARCHITECTURE + REPAIR DATA
# =========================================================

import time
import joblib

from sklearn.pipeline import (
    Pipeline,
    FeatureUnion
)

from sklearn.feature_extraction.text import (
    TfidfVectorizer
)

from sklearn.multiclass import (
    OneVsRestClassifier
)

from sklearn.linear_model import (
    LogisticRegression
)


# =========================================================
# DATA
# =========================================================

X_train_v2 = (
    train_v2_df[
        "command"
    ]
    .astype(str)
    .tolist()
)


X_valid_v2 = (
    valid_v2_df[
        "command"
    ]
    .astype(str)
    .tolist()
)


X_blind_v2 = (
    blind_v2_df[
        "command"
    ]
    .astype(str)
    .tolist()
)


# =========================================================
# FEATURES
# =========================================================

features_v2 = FeatureUnion(
    [

        (
            "word",

            TfidfVectorizer(

                analyzer="word",

                ngram_range=(
                    1,
                    3
                ),

                min_df=2,

                max_df=0.995,

                sublinear_tf=True,

                max_features=50000,
            )
        ),

        (
            "char",

            TfidfVectorizer(

                analyzer="char_wb",

                ngram_range=(
                    3,
                    6
                ),

                min_df=2,

                sublinear_tf=True,

                max_features=70000,
            )
        ),
    ]
)


# =========================================================
# CLASSIFIER
# =========================================================

classifier_v2 = OneVsRestClassifier(

    LogisticRegression(

        C=4.0,

        max_iter=2000,

        class_weight="balanced",

        solver="liblinear",

        random_state=SEED,
    ),

    n_jobs=-1
)


# =========================================================
# PIPELINE
# =========================================================

model_v2 = Pipeline(
    [
        (
            "features",
            features_v2
        ),

        (
            "classifier",
            classifier_v2
        ),
    ]
)


# =========================================================
# TRAIN
# =========================================================

print()
print("=" * 80)
print("ACTION CLASSIFIER V2 — TRAINING")
print("=" * 80)


start = time.time()


model_v2.fit(
    X_train_v2,
    y_train_v2
)


training_time_v2 = (
    time.time()
    - start
)


print(
    "Training complete"
)

print(
    "Training time:",
    round(
        training_time_v2,
        2
    ),
    "sec"
)


# =========================================================
# SAVE
# =========================================================

MODEL_V2_PATH = (
    f"{MODEL_DIR}/action_classifier_v2.joblib"
)

MLB_V2_PATH = (
    f"{MODEL_DIR}/multilabel_binarizer_v2.joblib"
)


joblib.dump(
    model_v2,
    MODEL_V2_PATH
)

joblib.dump(
    mlb_v2,
    MLB_V2_PATH
)


print()
print(
    "Model:",
    MODEL_V2_PATH
)

print(
    "MLB:",
    MLB_V2_PATH
)

print("=" * 80)


ACTION CLASSIFIER V2 — TRAINING
Training complete
Training time: 3.88 sec

Model: /content/drive/MyDrive/action_classifier/models/action_classifier_v2.joblib
MLB: /content/drive/MyDrive/action_classifier/models/multilabel_binarizer_v2.joblib


In [51]:
# =========================================================
# CELL 36 — V2 VALIDATION EVALUATION
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    hamming_loss,
    classification_report,
    multilabel_confusion_matrix,
)

import pandas as pd


# =========================================================
# PREDICT VALIDATION
# =========================================================

y_valid_pred_v2 = (
    model_v2.predict(
        X_valid_v2
    )
)


# =========================================================
# GLOBAL METRICS
# =========================================================

v2_subset_accuracy = accuracy_score(
    y_valid_v2,
    y_valid_pred_v2
)


v2_micro_precision = precision_score(
    y_valid_v2,
    y_valid_pred_v2,
    average="micro",
    zero_division=0
)


v2_micro_recall = recall_score(
    y_valid_v2,
    y_valid_pred_v2,
    average="micro",
    zero_division=0
)


v2_micro_f1 = f1_score(
    y_valid_v2,
    y_valid_pred_v2,
    average="micro",
    zero_division=0
)


v2_macro_f1 = f1_score(
    y_valid_v2,
    y_valid_pred_v2,
    average="macro",
    zero_division=0
)


v2_hamming = hamming_loss(
    y_valid_v2,
    y_valid_pred_v2
)


# =========================================================
# SUMMARY
# =========================================================

print()
print("=" * 80)
print("ACTION CLASSIFIER V2 — VALIDATION")
print("=" * 80)

print(
    "Validation samples:",
    len(
        X_valid_v2
    )
)

print(
    "Subset accuracy   :",
    round(
        v2_subset_accuracy,
        4
    )
)

print(
    "Micro precision   :",
    round(
        v2_micro_precision,
        4
    )
)

print(
    "Micro recall      :",
    round(
        v2_micro_recall,
        4
    )
)

print(
    "Micro F1          :",
    round(
        v2_micro_f1,
        4
    )
)

print(
    "Macro F1          :",
    round(
        v2_macro_f1,
        4
    )
)

print(
    "Hamming loss      :",
    round(
        v2_hamming,
        4
    )
)

print("=" * 80)


# =========================================================
# CLASSIFICATION REPORT
# =========================================================

print()

print(
    classification_report(
        y_valid_v2,
        y_valid_pred_v2,
        target_names=LABELS,
        zero_division=0
    )
)


# =========================================================
# CONFUSION
# =========================================================

mcm_v2 = multilabel_confusion_matrix(
    y_valid_v2,
    y_valid_pred_v2
)


v2_confusion_rows = []


for i, label in enumerate(
    LABELS
):

    tn, fp, fn, tp = (
        mcm_v2[
            i
        ].ravel()
    )


    v2_confusion_rows.append(
        {
            "label":
                label,

            "TP":
                int(tp),

            "FP":
                int(fp),

            "FN":
                int(fn),

            "TN":
                int(tn),
        }
    )


v2_confusion_df = (
    pd.DataFrame(
        v2_confusion_rows
    )
)


print()
print("=" * 80)
print("V2 — PER LABEL CONFUSION")
print("=" * 80)

display(
    v2_confusion_df
)


ACTION CLASSIFIER V2 — VALIDATION
Validation samples: 523
Subset accuracy   : 0.5985
Micro precision   : 0.843
Micro recall      : 0.8404
Micro F1          : 0.8417
Macro F1          : 0.8414
Hamming loss      : 0.0743

               precision    recall  f1-score   support

         READ       0.94      0.86      0.90       324
        WRITE       0.82      0.83      0.82       135
       DELETE       0.98      0.90      0.94        61
      EXECUTE       0.64      0.76      0.69       141
      NETWORK       0.70      0.85      0.77        75
      INSTALL       0.74      0.85      0.79        20
   PRIVILEGED       0.98      0.93      0.96        58
SYSTEM_CHANGE       0.90      0.82      0.86       170

    micro avg       0.84      0.84      0.84       984
    macro avg       0.84      0.85      0.84       984
 weighted avg       0.86      0.84      0.85       984
  samples avg       0.83      0.84      0.82       984


V2 — PER LABEL CONFUSION


,label,TP,FP,FN,TN
0,READ,279,19,45,180
1,WRITE,112,25,23,363
2,DELETE,55,1,6,461
3,EXECUTE,107,60,34,322
4,NETWORK,64,27,11,421
5,INSTALL,17,6,3,497
6,PRIVILEGED,54,1,4,464
7,SYSTEM_CHANGE,139,15,31,338


In [53]:
# =========================================================
# CELL 37 — VALIDATION PROBABILITY EXTRACTION
# MODEL 01 — ACTION CLASSIFIER V2
# =========================================================

import numpy as np


# =========================================================
# PROBABILITIES
# =========================================================

y_valid_prob_v2 = (
    model_v2.predict_proba(
        X_valid_v2
    )
)


print()
print("=" * 80)
print("ACTION CLASSIFIER V2 — VALIDATION PROBABILITIES")
print("=" * 80)

print(
    "Shape:",
    y_valid_prob_v2.shape
)

print(
    "Samples:",
    len(
        y_valid_prob_v2
    )
)

print(
    "Labels:",
    len(
        LABELS
    )
)

print("=" * 80)


ACTION CLASSIFIER V2 — VALIDATION PROBABILITIES
Shape: (523, 8)
Samples: 523
Labels: 8


In [54]:
# =========================================================
# CELL 38 — PER-LABEL THRESHOLD SEARCH
# MODEL 01 — ACTION CLASSIFIER V2
# =========================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
)


# =========================================================
# SEARCH SPACE
# =========================================================

THRESHOLD_GRID = np.arange(
    0.20,
    0.81,
    0.01
)


threshold_results = []


# =========================================================
# SEARCH
# =========================================================

for label_index, label in enumerate(
    LABELS
):

    y_true_label = (
        y_valid_v2[
            :,
            label_index
        ]
    )

    probability = (
        y_valid_prob_v2[
            :,
            label_index
        ]
    )


    best_result = None


    for threshold in THRESHOLD_GRID:

        predicted = (
            probability
            >= threshold
        ).astype(int)


        precision = precision_score(
            y_true_label,
            predicted,
            zero_division=0
        )

        recall = recall_score(
            y_true_label,
            predicted,
            zero_division=0
        )

        f1 = f1_score(
            y_true_label,
            predicted,
            zero_division=0
        )


        result = {

            "label":
                label,

            "threshold":
                float(
                    threshold
                ),

            "precision":
                float(
                    precision
                ),

            "recall":
                float(
                    recall
                ),

            "f1":
                float(
                    f1
                ),
        }


        if (
            best_result is None
            or
            f1
            >
            best_result[
                "f1"
            ]
        ):

            best_result = (
                result
            )


    threshold_results.append(
        best_result
    )


threshold_v2_df = (
    pd.DataFrame(
        threshold_results
    )
)


print()
print("=" * 80)
print("ACTION CLASSIFIER V2 — BEST VALIDATION THRESHOLDS")
print("=" * 80)

display(
    threshold_v2_df
)

print("=" * 80)


ACTION CLASSIFIER V2 — BEST VALIDATION THRESHOLDS


,label,threshold,precision,recall,f1
0,READ,0.39,0.909657,0.901235,0.905426
1,WRITE,0.51,0.829630,0.829630,0.829630
2,DELETE,0.36,0.950000,0.934426,0.942149
3,EXECUTE,0.58,0.731884,0.716312,0.724014
4,NETWORK,0.64,0.783784,0.773333,0.778523
5,INSTALL,0.63,0.850000,0.850000,0.850000
6,PRIVILEGED,0.44,0.982143,0.948276,0.964912
7,SYSTEM_CHANGE,0.37,0.850000,0.900000,0.874286


In [55]:
# =========================================================
# CELL 39 — VALIDATION EVALUATION WITH TUNED THRESHOLDS
# MODEL 01 — ACTION CLASSIFIER V2
# =========================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    classification_report,
    multilabel_confusion_matrix,
)


# =========================================================
# BUILD THRESHOLD VECTOR
# =========================================================

threshold_vector = np.array(
    [
        threshold_v2_df.loc[
            threshold_v2_df[
                "label"
            ] == label,
            "threshold"
        ].iloc[0]

        for label in LABELS
    ]
)


print(
    "Threshold vector:"
)

for label, threshold in zip(
    LABELS,
    threshold_vector
):

    print(
        f"{label:14}:",
        round(
            float(
                threshold
            ),
            2
        )
    )


# =========================================================
# APPLY THRESHOLDS
# =========================================================

y_valid_pred_tuned = (
    y_valid_prob_v2
    >=
    threshold_vector
).astype(int)


# =========================================================
# GLOBAL METRICS
# =========================================================

tuned_subset_accuracy = accuracy_score(
    y_valid_v2,
    y_valid_pred_tuned
)


tuned_micro_f1 = f1_score(
    y_valid_v2,
    y_valid_pred_tuned,
    average="micro",
    zero_division=0
)


tuned_macro_f1 = f1_score(
    y_valid_v2,
    y_valid_pred_tuned,
    average="macro",
    zero_division=0
)


tuned_micro_precision = precision_score(
    y_valid_v2,
    y_valid_pred_tuned,
    average="micro",
    zero_division=0
)


tuned_micro_recall = recall_score(
    y_valid_v2,
    y_valid_pred_tuned,
    average="micro",
    zero_division=0
)


tuned_hamming = hamming_loss(
    y_valid_v2,
    y_valid_pred_tuned
)


print()
print("=" * 80)
print("V2 — TUNED VALIDATION METRICS")
print("=" * 80)

print(
    "Subset accuracy :",
    round(
        tuned_subset_accuracy,
        4
    )
)

print(
    "Micro precision :",
    round(
        tuned_micro_precision,
        4
    )
)

print(
    "Micro recall    :",
    round(
        tuned_micro_recall,
        4
    )
)

print(
    "Micro F1        :",
    round(
        tuned_micro_f1,
        4
    )
)

print(
    "Macro F1        :",
    round(
        tuned_macro_f1,
        4
    )
)

print(
    "Hamming loss    :",
    round(
        tuned_hamming,
        4
    )
)

print("=" * 80)


print()

print(
    classification_report(
        y_valid_v2,
        y_valid_pred_tuned,
        target_names=LABELS,
        zero_division=0
    )
)

Threshold vector:
READ          : 0.39
WRITE         : 0.51
DELETE        : 0.36
EXECUTE       : 0.58
NETWORK       : 0.64
INSTALL       : 0.63
PRIVILEGED    : 0.44
SYSTEM_CHANGE : 0.37

V2 — TUNED VALIDATION METRICS
Subset accuracy : 0.631
Micro precision : 0.8587
Micro recall    : 0.8587
Micro F1        : 0.8587
Macro F1        : 0.8586
Hamming loss    : 0.0664

               precision    recall  f1-score   support

         READ       0.91      0.90      0.91       324
        WRITE       0.83      0.83      0.83       135
       DELETE       0.95      0.93      0.94        61
      EXECUTE       0.73      0.72      0.72       141
      NETWORK       0.78      0.77      0.78        75
      INSTALL       0.85      0.85      0.85        20
   PRIVILEGED       0.98      0.95      0.96        58
SYSTEM_CHANGE       0.85      0.90      0.87       170

    micro avg       0.86      0.86      0.86       984
    macro avg       0.86      0.86      0.86       984
 weighted avg       0.86  

In [58]:
# =========================================================
# CELL 40 — FREEZE THRESHOLD CONFIG
# MODEL 01 — ACTION CLASSIFIER V2
# =========================================================

import json


THRESHOLD_PATH = (
    f"{MODEL_DIR}/action_classifier_v2_thresholds.json"
)


threshold_config = {

    label:
        float(
            threshold
        )

    for label, threshold in zip(
        LABELS,
        threshold_vector
    )
}


with open(
    THRESHOLD_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        threshold_config,
        f,
        indent=2
    )


print()
print("=" * 80)
print("ACTION CLASSIFIER V2 — THRESHOLDS SAVED")
print("=" * 80)

print(
    json.dumps(
        threshold_config,
        indent=2
    )
)

print()

print(
    "Saved:",
    THRESHOLD_PATH
)

print("=" * 80)


ACTION CLASSIFIER V2 — THRESHOLDS SAVED
{
  "READ": 0.3900000000000002,
  "WRITE": 0.5100000000000002,
  "DELETE": 0.36000000000000015,
  "EXECUTE": 0.5800000000000003,
  "NETWORK": 0.6400000000000003,
  "INSTALL": 0.6300000000000003,
  "PRIVILEGED": 0.4400000000000002,
  "SYSTEM_CHANGE": 0.37000000000000016
}

Saved: /content/drive/MyDrive/action_classifier/models/action_classifier_v2_thresholds.json


In [59]:
# =========================================================
# CELL 41 — FINAL BLIND TEST
# MODEL 01 — ACTION CLASSIFIER V2
# =========================================================

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    classification_report,
    multilabel_confusion_matrix,
)


# =========================================================
# 1. PREDICT PROBABILITIES
# =========================================================

y_blind_prob_v2 = (
    model_v2.predict_proba(
        X_blind_v2
    )
)


# =========================================================
# 2. APPLY FROZEN THRESHOLDS
# =========================================================

y_blind_pred_v2 = (
    y_blind_prob_v2
    >= threshold_vector
).astype(int)


# =========================================================
# 3. GLOBAL METRICS
# =========================================================

blind_subset_accuracy = accuracy_score(
    y_blind_v2,
    y_blind_pred_v2
)

blind_micro_precision = precision_score(
    y_blind_v2,
    y_blind_pred_v2,
    average="micro",
    zero_division=0
)

blind_micro_recall = recall_score(
    y_blind_v2,
    y_blind_pred_v2,
    average="micro",
    zero_division=0
)

blind_micro_f1 = f1_score(
    y_blind_v2,
    y_blind_pred_v2,
    average="micro",
    zero_division=0
)

blind_macro_f1 = f1_score(
    y_blind_v2,
    y_blind_pred_v2,
    average="macro",
    zero_division=0
)

blind_hamming = hamming_loss(
    y_blind_v2,
    y_blind_pred_v2
)


# =========================================================
# 4. STATUS
# =========================================================

print()
print("=" * 80)
print("ACTION CLASSIFIER V2 — FINAL BLIND TEST")
print("=" * 80)

print(
    "Blind samples     :",
    len(X_blind_v2)
)

print(
    "Subset accuracy   :",
    round(blind_subset_accuracy, 4)
)

print(
    "Micro precision   :",
    round(blind_micro_precision, 4)
)

print(
    "Micro recall      :",
    round(blind_micro_recall, 4)
)

print(
    "Micro F1          :",
    round(blind_micro_f1, 4)
)

print(
    "Macro F1          :",
    round(blind_macro_f1, 4)
)

print(
    "Hamming loss      :",
    round(blind_hamming, 4)
)

print("=" * 80)


print()

print(
    classification_report(
        y_blind_v2,
        y_blind_pred_v2,
        target_names=LABELS,
        zero_division=0
    )
)


ACTION CLASSIFIER V2 — FINAL BLIND TEST
Blind samples     : 523
Subset accuracy   : 0.6157
Micro precision   : 0.8522
Micro recall      : 0.841
Micro F1          : 0.8466
Macro F1          : 0.8549
Hamming loss      : 0.0724

               precision    recall  f1-score   support

         READ       0.87      0.93      0.90       304
        WRITE       0.82      0.76      0.78       123
       DELETE       0.93      0.93      0.93        69
      EXECUTE       0.71      0.63      0.67       153
      NETWORK       0.83      0.76      0.80        85
      INSTALL       0.96      0.88      0.92        26
   PRIVILEGED       0.99      0.95      0.97        73
SYSTEM_CHANGE       0.86      0.89      0.87       161

    micro avg       0.85      0.84      0.85       994
    macro avg       0.87      0.84      0.85       994
 weighted avg       0.85      0.84      0.84       994
  samples avg       0.84      0.85      0.83       994



In [60]:
# =========================================================
# CELL 42 — BLIND CONFUSION MATRIX
# MODEL 01 — ACTION CLASSIFIER V2
# =========================================================

import pandas as pd


blind_mcm = multilabel_confusion_matrix(
    y_blind_v2,
    y_blind_pred_v2
)


blind_confusion_rows = []


for i, label in enumerate(LABELS):

    tn, fp, fn, tp = (
        blind_mcm[i].ravel()
    )

    blind_confusion_rows.append(
        {
            "label": label,
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "TN": int(tn),
        }
    )


blind_confusion_df = pd.DataFrame(
    blind_confusion_rows
)


print()
print("=" * 80)
print("ACTION CLASSIFIER V2 — BLIND CONFUSION")
print("=" * 80)

display(
    blind_confusion_df
)

print("=" * 80)


ACTION CLASSIFIER V2 — BLIND CONFUSION


,label,TP,FP,FN,TN
0,READ,282,42,22,177
1,WRITE,93,21,30,379
2,DELETE,64,5,5,449
3,EXECUTE,97,39,56,331
4,NETWORK,65,13,20,425
5,INSTALL,23,1,3,496
6,PRIVILEGED,69,1,4,449
7,SYSTEM_CHANGE,143,23,18,339


In [61]:
# =========================================================
# CELL 43 — INTERACTIVE PREDICT
# MODEL 01 — ACTION CLASSIFIER V2
# =========================================================

import numpy as np


def predict_actions(
    command,
    show_probabilities=True
):

    command = str(
        command
    ).strip()


    if not command:

        raise ValueError(
            "Command kosong."
        )


    probabilities = (
        model_v2.predict_proba(
            [command]
        )[0]
    )


    predictions = (
        probabilities
        >= threshold_vector
    )


    actions = [
        LABELS[i]
        for i, active in enumerate(
            predictions
        )
        if active
    ]


    result = {
        "command": command,
        "actions": actions,
    }


    if show_probabilities:

        result[
            "probabilities"
        ] = {

            LABELS[i]:
                round(
                    float(
                        probabilities[i]
                    ),
                    4
                )

            for i in range(
                len(LABELS)
            )
        }


    return result

In [62]:
tests = [
    "cat /etc/os-release",
    "rm test.txt",
    "curl https://example.com/file -o file",
    "python app.py",
    "sudo systemctl restart nginx",
    "sudo apt install nginx",
]


for command in tests:

    result = predict_actions(
        command
    )

    print()
    print("=" * 70)

    print(
        "COMMAND:",
        command
    )

    print(
        "ACTIONS:",
        result["actions"]
    )

    print(
        "PROBS:",
        result["probabilities"]
    )


COMMAND: cat /etc/os-release
ACTIONS: ['READ']
PROBS: {'READ': 0.9063, 'WRITE': 0.1004, 'DELETE': 0.0206, 'EXECUTE': 0.1178, 'NETWORK': 0.0669, 'INSTALL': 0.0105, 'PRIVILEGED': 0.0371, 'SYSTEM_CHANGE': 0.2973}

COMMAND: rm test.txt
ACTIONS: ['DELETE']
PROBS: {'READ': 0.1512, 'WRITE': 0.2979, 'DELETE': 0.8568, 'EXECUTE': 0.0381, 'NETWORK': 0.0801, 'INSTALL': 0.012, 'PRIVILEGED': 0.0583, 'SYSTEM_CHANGE': 0.0471}

COMMAND: curl https://example.com/file -o file
ACTIONS: ['WRITE', 'NETWORK']
PROBS: {'READ': 0.0882, 'WRITE': 0.8688, 'DELETE': 0.0086, 'EXECUTE': 0.1996, 'NETWORK': 0.9975, 'INSTALL': 0.0116, 'PRIVILEGED': 0.0103, 'SYSTEM_CHANGE': 0.0243}

COMMAND: python app.py
ACTIONS: ['EXECUTE']
PROBS: {'READ': 0.0774, 'WRITE': 0.2729, 'DELETE': 0.02, 'EXECUTE': 0.9326, 'NETWORK': 0.1252, 'INSTALL': 0.0232, 'PRIVILEGED': 0.0318, 'SYSTEM_CHANGE': 0.0673}

COMMAND: sudo systemctl restart nginx
ACTIONS: ['EXECUTE', 'PRIVILEGED', 'SYSTEM_CHANGE']
PROBS: {'READ': 0.0338, 'WRITE': 0.0276, 'DELET

In [63]:
# =========================================================
# CELL 44 — SAVE FINAL V2 METRICS
# MODEL 01 — ACTION CLASSIFIER
# =========================================================

import json


FINAL_V2_METRICS_PATH = (
    f"{REPORT_DIR}/action_classifier_v2_final_metrics.json"
)


final_metrics = {

    "validation": {

        "subset_accuracy":
            float(
                tuned_subset_accuracy
            ),

        "micro_f1":
            float(
                tuned_micro_f1
            ),

        "macro_f1":
            float(
                tuned_macro_f1
            ),

        "hamming_loss":
            float(
                tuned_hamming
            ),
    },


    "blind": {

        "subset_accuracy":
            float(
                blind_subset_accuracy
            ),

        "micro_precision":
            float(
                blind_micro_precision
            ),

        "micro_recall":
            float(
                blind_micro_recall
            ),

        "micro_f1":
            float(
                blind_micro_f1
            ),

        "macro_f1":
            float(
                blind_macro_f1
            ),

        "hamming_loss":
            float(
                blind_hamming
            ),
    },


    "thresholds":
        threshold_config,
}


with open(
    FINAL_V2_METRICS_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_metrics,
        f,
        indent=2
    )


print()
print("=" * 80)
print("ACTION CLASSIFIER V2 — FINAL METRICS SAVED")
print("=" * 80)

print(
    FINAL_V2_METRICS_PATH
)

print("=" * 80)


ACTION CLASSIFIER V2 — FINAL METRICS SAVED
/content/drive/MyDrive/action_classifier/reports/action_classifier_v2_final_metrics.json


In [64]:
# =========================================================
# CELL 44.1 — RANDOM EXECUTE STRESS TEST
# MODEL 01 — ACTION CLASSIFIER V2
# =========================================================

RANDOM_EXECUTE_TESTS = [

    # =====================================================
    # SHOULD BE EXECUTE
    # =====================================================

    "python3 script.py",

    "bash deploy.sh",

    "sh ./installer.sh",

    "node server.js",

    "perl cleanup.pl",

    "ruby app.rb",

    "./my_program",

    "python -m http.server 8000",

    "bash -c 'echo hello'",

    "env python3 worker.py",

    "nohup python3 worker.py > worker.log 2>&1 &",

    "sudo systemctl restart ssh",

    "sudo service nginx restart",


    # =====================================================
    # READ — SHOULD NOT BECOME EXECUTE
    # =====================================================

    "cat /var/log/syslog",

    "grep -R 'ERROR' /var/log",

    "find /tmp -type f -name '*.log'",

    "head -n 20 app.log",

    "tail -f nginx.log",

    "ps aux",

    "df -h",

    "du -sh /home/user",

    "stat /etc/passwd",

    "whoami",


    # =====================================================
    # WRITE — SHOULD NOT AUTOMATICALLY BECOME EXECUTE
    # =====================================================

    "touch output.txt",

    "cp source.txt backup.txt",

    "mv old.txt new.txt",

    "echo hello > output.txt",

    "printf 'hello' >> log.txt",

    "mkdir -p /tmp/example",


    # =====================================================
    # FIND -EXEC BOUNDARY
    # =====================================================

    "find /tmp -type f -exec chmod 600 {} \\;",

    "find /home -name '*.txt' -exec cp {} /tmp \\;",

    "find /tmp -type f -exec rm {} \\;",

    "find . -type f -exec cat {} \\;",


    # =====================================================
    # NETWORK VS EXECUTE
    # =====================================================

    "curl https://example.com",

    "wget https://example.com/file -O file",

    "ssh user@example.com",

    "scp file.txt user@example.com:/tmp/",


    # =====================================================
    # INSTALL
    # =====================================================

    "pip install flask",

    "npm install express",

    "sudo apt install nginx",

    "sudo apt update",


    # =====================================================
    # CHAINED COMMANDS
    # =====================================================

    "curl https://example.com/script.py -o script.py && python script.py",

    "cat input.txt | python parser.py",

    "grep ERROR app.log && python notify.py",

    "cp config.ini backup.ini && systemctl restart app",

    "wget https://example.com/app.sh -O app.sh && bash app.sh",
]


print()
print("=" * 100)
print("ACTION CLASSIFIER V2 — RANDOM EXECUTE STRESS TEST")
print("=" * 100)


stress_results = []


for i, command in enumerate(
    RANDOM_EXECUTE_TESTS,
    start=1
):

    result = predict_actions(
        command,
        show_probabilities=True
    )

    execute_probability = (
        result[
            "probabilities"
        ][
            "EXECUTE"
        ]
    )


    stress_results.append(
        {
            "id": i,
            "command": command,
            "actions": result["actions"],
            "execute_probability":
                execute_probability,
        }
    )


    print()
    print(
        f"[{i:02}/{len(RANDOM_EXECUTE_TESTS)}]"
    )

    print(
        "COMMAND :",
        command
    )

    print(
        "ACTIONS :",
        result[
            "actions"
        ]
    )

    print(
        "EXECUTE :",
        execute_probability,
        "| threshold:",
        round(
            threshold_config[
                "EXECUTE"
            ],
            2
        )
    )

    print("-" * 100)


stress_test_df = pd.DataFrame(
    stress_results
)


print()
print("=" * 100)
print("STRESS TEST COMPLETE")
print("=" * 100)

print(
    "Samples:",
    len(
        stress_test_df
    )
)

print("=" * 100)


ACTION CLASSIFIER V2 — RANDOM EXECUTE STRESS TEST

[01/46]
COMMAND : python3 script.py
ACTIONS : ['EXECUTE']
EXECUTE : 0.9106 | threshold: 0.58
----------------------------------------------------------------------------------------------------

[02/46]
COMMAND : bash deploy.sh
ACTIONS : ['EXECUTE', 'NETWORK']
EXECUTE : 0.9953 | threshold: 0.58
----------------------------------------------------------------------------------------------------

[03/46]
COMMAND : sh ./installer.sh
ACTIONS : ['EXECUTE']
EXECUTE : 0.9911 | threshold: 0.58
----------------------------------------------------------------------------------------------------

[04/46]
COMMAND : node server.js
ACTIONS : ['READ', 'EXECUTE']
EXECUTE : 0.5806 | threshold: 0.58
----------------------------------------------------------------------------------------------------

[05/46]
COMMAND : perl cleanup.pl
ACTIONS : ['EXECUTE']
EXECUTE : 0.7592 | threshold: 0.58
----------------------------------------------------------------

In [65]:
# =========================================================
# CELL 44.2 — EXECUTE PREDICTION SUMMARY
# =========================================================

execute_predicted_df = (
    stress_test_df[
        stress_test_df[
            "actions"
        ].apply(
            lambda x:
                "EXECUTE" in x
        )
    ]
    .copy()
)


non_execute_predicted_df = (
    stress_test_df[
        ~stress_test_df[
            "actions"
        ].apply(
            lambda x:
                "EXECUTE" in x
        )
    ]
    .copy()
)


print()
print("=" * 100)
print("PREDICTED AS EXECUTE")
print("=" * 100)

display(
    execute_predicted_df[
        [
            "command",
            "actions",
            "execute_probability",
        ]
    ]
)


print()
print("=" * 100)
print("NOT PREDICTED AS EXECUTE")
print("=" * 100)

display(
    non_execute_predicted_df[
        [
            "command",
            "actions",
            "execute_probability",
        ]
    ]
)


PREDICTED AS EXECUTE


,command,actions,execute_probability
0,python3 script.py,[EXECUTE],0.9106
1,bash deploy.sh,"[EXECUTE, NETWORK]",0.9953
2,sh ./installer.sh,[EXECUTE],0.9911
3,node server.js,"[READ, EXECUTE]",0.5806
4,perl cleanup.pl,[EXECUTE],0.7592
5,ruby app.rb,[EXECUTE],0.7161
6,./my_program,[EXECUTE],0.7778
7,python -m http.server 8000,"[EXECUTE, NETWORK]",0.8264
8,bash -c 'echo hello',[EXECUTE],0.9067
9,env python3 worker.py,[EXECUTE],0.9346



NOT PREDICTED AS EXECUTE


,command,actions,execute_probability
13,cat /var/log/syslog,[READ],0.2906
14,grep -R 'ERROR' /var/log,[READ],0.0855
15,find /tmp -type f -name '*.log',[READ],0.0198
16,head -n 20 app.log,[READ],0.5003
17,tail -f nginx.log,[READ],0.0973
19,df -h,[READ],0.3007
20,du -sh /home/user,[READ],0.2212
21,stat /etc/passwd,"[READ, SYSTEM_CHANGE]",0.0536
22,whoami,[READ],0.2755
23,touch output.txt,[WRITE],0.3041


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.
